# Event test: download, extract and check every value

Runs the whole pipeline on **one GDACS event** and then checks that every value of the final
dataset agrees with what the APIs returned.

**How to use it**

1. Open it from the project folder, in the environment with the libraries (`env_displaced`), with the keys in `.env`.
2. Edit the settings in section 0 (the default event is EQ_1496100, Indonesia, 16 August 2025).
3. Close `dataset_variables.xlsx`, then run all the cells in order. A failing check does not stop the notebook.
4. Read the summary in section 15. All the results are also saved in `test_event_results.csv`.

Every check prints one line:

- **PASS**: it works, or the two values are equal.
- **FAIL**: it does not work, or the dataset differs from the original file. This is an error to fix.
- **WARN**: something to read, not an error. Two sources disagree, or GDACS changed a file after it was saved.
- **SKIP**: it could not run, for example a missing key or no data for this country. The reason is printed.

**What changes on disk**

- Section 2 moves the saved event folder to `backup/<EVENT>_<time>/`, so the downloads start from zero. It also copies `dataset/<ISO3>/<YEAR>/` and `output/<ISO3>/<YEAR>/country/` there, for the before/after comparison.
- The downloads write in `output/`, and section 13 rebuilds `dataset/` for the country of the event, as the normal workflow does.
- With `RUN_DOWNLOADS = False` nothing is downloaded or moved: the notebook rebuilds the dataset from what is already saved, then checks it.

**How the checks work (section 14)**

The checks never use the extraction code of `sources/dataset/`. They read the original files with `json` and `pandas`, rebuild every value following the rules written in `scelte_metodologiche.md`, and compare it with the dataset:

| Section | Question |
|---|---|
| 14a | Is every value of the long tables the one in the original file? |
| 14b | Does every number of the long tables appear in the file it says it comes from? |
| 14c | Is every cell of the tables in `dataset/<ISO3>/<YEAR>/` the one the long tables give? |
| 14d | Do the grid cells add up to the location values? |
| 14e | Totals, shares, dates and versions: are they consistent? |
| 14f | Do the sources agree with each other? |
| 14g | What changed from the dataset before this run? |
| 14h | Values to compare by hand on the websites. |
| 14i | Which columns were checked, and which were not? |

## 0 - Settings

In [ ]:
EVENTTYPE = "EQ"
EVENTID = 1496100              # Earthquake in Indonesia, 16 August 2025

RUN_DOWNLOADS = True           # False: download nothing, check what is already saved
DO_BACKUP = True               # move the saved event folder to backup/ before downloading it again
CHECK_LIVE_API = True          # call GDACS again and compare the answers with the saved files

NEWS_LIMIT = 50                # EMM news to download
RELIEFWEB_DAYS = 30            # ReliefWeb reports up to N days after the event
GLIDE = None                   # GLIDE to try for ReliefWeb, e.g. "EQ-2025-000123-IDN"; None = find it

RUN_GHSL = True                # population and settlement class of every area of the event
RUN_GHSL_FULL = False          # True: GHSL of every admin unit of the country (slow, saved).
                               # False: only the province of the epicentre, checked and not saved
POP_TIF = None                 # None: rasters found through RELEASE_ROOT in sources/ghsl/ghsl.py
SMOD_TIF = None
RUN_MONTANDON = True

REL = 1e-6                     # dataset against the original file
REL_RASTER = 0.005             # exactextract against rasterio (cells on the edge can differ)
REL_AREA = 0.005               # area of the project formula against pyproj
REL_SOURCES = 0.25             # two sources: a larger difference is a WARN
MAX_PROVENANCE_MB = 400        # 14b: larger original files are not searched

### Helpers

In [ ]:
import json
import math
import os
import platform
import re
import shutil
import unicodedata
import warnings
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

RESULTS = []          # one row per check
VERIFIED = {}         # (table level, column) -> how its cells were checked (14c, 14d)
RAW_CHECKED = set()   # (level, source, variable) compared with the original files (14a)

try:
    display
except NameError:     # outside Jupyter
    display = print


class Skip(Exception):
    """Raised to skip the rest of a block, with the reason."""


def record(section, name, status, detail=""):
    RESULTS.append({"section": section, "check": name, "status": status, "detail": str(detail)[:800]})
    print(f"[{status}] {section} | {name}" + (f" | {detail}" if str(detail) != "" else ""))


def check(section, name, ok, detail=""):
    """PASS or FAIL."""
    record(section, name, "PASS" if ok else "FAIL", detail)
    return bool(ok)


def warn(section, name, ok, detail=""):
    """PASS or WARN: a difference to read, not an error."""
    record(section, name, "PASS" if ok else "WARN", detail)
    return bool(ok)


def missing(v):
    if v is None or v is pd.NaT:
        return True
    try:
        return bool(pd.isna(v))
    except (TypeError, ValueError):
        return False


def is_number(v):
    if missing(v) or isinstance(v, bool):
        return False
    try:
        return not math.isnan(float(v))
    except (TypeError, ValueError):
        return False


def equal(got, expected, rel=REL):
    if is_number(got) and is_number(expected):
        g, e = float(got), float(expected)
        return abs(g - e) <= rel * max(1.0, abs(e))
    return str(got) == str(expected)


def same(section, name, got, expected, rel=REL, strict=True):
    """Two values must be equal (numbers within `rel`). strict=False gives WARN instead of FAIL."""
    out = check if strict else warn
    gaps = [w for w, v in (("dataset", got), ("original", expected)) if missing(v)]
    if gaps:
        return out(section, name, False, f"missing value in the {' and the '.join(gaps)}: {got!r} / {expected!r}")
    if is_number(got) and is_number(expected):
        shown = f"dataset {float(got):,.6g}, original {float(expected):,.6g}"
    else:
        shown = f"dataset {got!r}, original {expected!r}"
    return out(section, name, equal(got, expected, rel), shown)


def explain(title, **values):
    """Print what is being looked at, one value per line."""
    print(f"\n--- {title}")
    width = max((len(k) for k in values), default=0)
    for k, v in values.items():
        print(f"    {k:<{width}} : {v}")


@contextmanager
def block(section, name):
    """Runs a block: an error becomes a FAIL and the notebook goes on."""
    try:
        yield
    except Skip as e:
        record(section, name, "SKIP", e)
    except Exception as e:
        record(section, name, "FAIL", f"{type(e).__name__}: {e}")


def need_downloads():
    if not RUN_DOWNLOADS:
        raise Skip("RUN_DOWNLOADS is False")


def need_key(*names):
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    gaps = [n for n in names if not os.getenv(n)]
    if gaps:
        raise Skip(f"missing in .env: {', '.join(gaps)}")


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def size_of(path):
    p = Path(path)
    return p.stat().st_size if p.is_file() else 0


def saved(section, path, what):
    n = size_of(path)
    return check(section, f"{what} saved", n > 0, f"{Path(path).as_posix()} ({n:,} bytes)" if n else f"{Path(path).as_posix()} missing or empty")


def diff_paths(a, b, path="", out=None, limit=20):
    """Places where two JSON objects differ (at most `limit`)."""
    out = [] if out is None else out
    if len(out) >= limit:
        return out
    if isinstance(a, dict) and isinstance(b, dict):
        for k in sorted(set(a) | set(b), key=str):
            diff_paths(a.get(k), b.get(k), f"{path}.{k}" if path else str(k), out, limit)
    elif isinstance(a, list) and isinstance(b, list) and len(a) == len(b):
        for i, (x, y) in enumerate(zip(a, b)):
            diff_paths(x, y, f"{path}[{i}]", out, limit)
    elif a != b:
        out.append(path or "<whole file>")
    return out


_NUMBER = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_NUMBERS = {}


def numbers_in(path):
    """Every number of a JSON file, also the ones written inside strings (sorted, unique)."""
    key = str(path)
    if key not in _NUMBERS:
        found = []

        def walk(o):
            if o is None or isinstance(o, bool):
                return
            if isinstance(o, (int, float)):
                found.append(float(o))
            elif isinstance(o, str):
                found.extend(float(x) for x in _NUMBER.findall(o))
            elif isinstance(o, dict):
                for v in o.values():
                    walk(v)
            elif isinstance(o, list):
                for v in o:
                    walk(v)

        walk(load_json(path))
        _NUMBERS[key] = np.unique(np.array(found, dtype=float))
    return _NUMBERS[key]


def in_file(value, path, rel=REL):
    arr = numbers_in(path)
    if not len(arr):
        return False
    i = int(np.searchsorted(arr, value))
    near = arr[max(0, i - 1): i + 1]
    return bool(np.any(np.abs(near - value) <= rel * max(1.0, abs(value))))


def first_number(text):
    """'10 Km' -> 10.0, '0.28 m' -> 0.28; None if there is no number."""
    m = re.search(r"[-+]?\d+(?:\.\d+)?", str(text if text is not None else ""))
    return float(m.group()) if m else None


def slug(text):
    """Variable names: ASCII, lower case, '_' between words."""
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def label_of(text):
    """Folder name of an area: 'Intensity 4.5' -> 'Intensity_4.5'."""
    text = re.sub(r"\s+", "_", str(text).strip())
    return re.sub(r"[^A-Za-z0-9._-]", "", text) or "feature"


def as_text(v):
    """Booleans as true/false, as the dataset writes them."""
    if isinstance(v, bool) or str(v) in ("True", "False"):
        return str(v).lower()
    return None if v is None or str(v).strip() == "" else str(v)


def haversine_km(lat1, lon1, lat2, lon2):
    a, b, c, d = map(math.radians, (lat1, lon1, lat2, lon2))
    h = math.sin((c - a) / 2) ** 2 + math.cos(a) * math.cos(c) * math.sin((d - b) / 2) ** 2
    return 6371.0 * 2 * math.asin(math.sqrt(h))


def episodes_of(props):
    """Episode ids of an event: those in episodes[].details plus the main one."""
    ids = {int(x) for e in props.get("episodes") or [] if isinstance(e, dict)
           for x in re.findall(r"episodeid=(\d+)", str(e.get("details") or ""))}
    if props.get("episodeid") is not None:
        ids.add(int(props["episodeid"]))
    return sorted(ids)

### Comparing the dataset with the original files

`compare` takes the rows of one source from the long tables and the rows rebuilt from the
original files, matches them on their keys and prints one line per variable:

- values that differ,
- values in the original file but missing from the dataset,
- values in the dataset that the original file does not give.

A variable that the check does not rebuild is a WARN: it is listed in section 14i as not verified.

In [ ]:
META = ("known_at", "valid_from", "valid_to")


def _key(v):
    if missing(v):
        return ""
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)


def _ts(v):
    if missing(v):
        return None
    t = pd.to_datetime(v, errors="coerce", utc=True, format="ISO8601") if isinstance(v, str) else pd.to_datetime(v, errors="coerce", utc=True)
    return None if missing(t) else t.tz_localize(None)


def compare(section, level, source, dataset_rows, expected, keys=(), rel=REL, rels=None, label=None):
    """Dataset rows of one source against the rows rebuilt from the original files.

    expected: list of dicts with the `keys`, `variable` and either `value` (number) or `text`;
    optionally known_at, valid_from, valid_to, compared to the second.
    """
    rels = rels or {}
    label = label or source
    keys = list(keys)
    allk = [*keys, "variable"]
    E = pd.DataFrame(list(expected))
    for c in ("value", "text", *META, *allk):
        if c not in E:
            E[c] = None
    if len(E):
        E["text"] = E["text"].map(lambda v: None if missing(v) or str(v).strip() == "" else str(v))
        E = E[E["value"].map(is_number) | E["text"].notna()].copy()
    D = dataset_rows.copy() if len(dataset_rows) else pd.DataFrame(columns=[*allk, "value", "value_text", *META])
    for c in (*allk, "value", "value_text", *META):
        if c not in D:
            D[c] = None
    if D.empty and E.empty:
        raise Skip(f"no {label} values in the dataset and none in the original files")
    for df in (D, E):
        for k in allk:
            df[k] = df[k].map(_key)
    dup = D[D.duplicated(allk, keep=False)]
    if len(dup):
        check(section, f"{label}: one value per key", False,
              f"{len(dup)} rows share a key, e.g. {dup[allk].head(3).to_dict('records')}")
    E = E.rename(columns={"value": "value_exp", **{m: f"{m}_exp" for m in META}})
    m = D.merge(E[[*allk, "value_exp", "text", *[f"{x}_exp" for x in META]]], on=allk, how="outer", indicator=True)
    wanted = set(E["variable"])
    for var, g in m.groupby("variable", sort=True):
        only_d = g[g["_merge"] == "left_only"]
        if var not in wanted:
            record(section, f"{label}.{var}", "WARN", f"{len(only_d)} dataset values that this check does not rebuild")
            continue
        bad = []
        for r in g[g["_merge"] == "both"].to_dict("records"):
            if is_number(r["value_exp"]):
                ok = is_number(r.get("value")) and equal(r["value"], r["value_exp"], rels.get(var, rel))
                shown = [r.get("value"), r["value_exp"]]
            else:
                ok = str(r.get("value_text")) == str(r["text"])
                shown = [r.get("value_text"), r["text"]]
            for x in META:
                want = _ts(r.get(f"{x}_exp"))
                if want is not None:
                    got = _ts(r.get(x))
                    if got is None or abs((got - want).total_seconds()) > 1:
                        ok = False
                        shown.append(f"{x} {got} vs {want}")
            if not ok:
                bad.append({**{k: r[k] for k in keys}, "dataset / original": shown})
        only_e = g[g["_merge"] == "right_only"]
        n_both = int((g["_merge"] == "both").sum())
        if n_both:
            RAW_CHECKED.add((level, source, var))
        problems = []
        if bad:
            problems.append(f"{len(bad)} differ, e.g. {bad[:2]}")
        if len(only_e):
            problems.append(f"{len(only_e)} missing in the dataset, e.g. {only_e[keys + ['value_exp', 'text']].head(2).to_dict('records')}")
        if len(only_d):
            problems.append(f"{len(only_d)} only in the dataset, e.g. {only_d[keys + ['value', 'value_text']].head(2).to_dict('records')}")
        check(section, f"{label}.{var}", not problems, "; ".join(problems) or f"{n_both} values equal to the original")
    return m

## 1 - Environment

In [ ]:
S = "1 environment"
import importlib

check(S, "project folder", Path("sources/dataset/build.py").is_file(), Path.cwd())
print("Python", platform.python_version())
check(S, "pandas 2 or newer", int(pd.__version__.split(".")[0]) >= 2, pd.__version__)
for lib in ("numpy", "pyarrow", "openpyxl", "requests", "httpx", "dotenv", "geopandas", "pyogrio", "shapely",
            "pyproj", "rasterio", "exactextract", "pystac_client", "dtmapi", "pdfplumber"):
    with block(S, f"library {lib}"):
        importlib.import_module(lib)
        record(S, f"library {lib}", "PASS")

with block(S, "keys"):
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    for key in ("IDMC_CLIENT_ID", "DTMAPI_SUBSCRIPTION_KEY", "MONTANDON_API_TOKEN"):
        record(S, f"key {key}", "PASS" if os.getenv(key) else "SKIP", "" if os.getenv(key) else "not in .env")
    hdx_key = os.getenv("HDX_APP_IDENTIFIER") or (os.getenv("HDX_APP_NAME") and os.getenv("HDX_APP_EMAIL"))
    record(S, "key HDX", "PASS" if hdx_key else "SKIP", "" if hdx_key else "not in .env")

with block(S, "GHSL rasters"):
    from sources.ghsl.ghsl import RELEASE_ROOT
    ok = bool(POP_TIF and SMOD_TIF) or RELEASE_ROOT.is_dir()
    record(S, "GHSL rasters", "PASS" if ok else "SKIP",
           POP_TIF or RELEASE_ROOT if ok else f"{RELEASE_ROOT} not found: set POP_TIF and SMOD_TIF, or GHSL is skipped")

## 2 - Event folder, backup and snapshot

GDACS gives the country and the year of the event, which decide its folder
`output/<ISO3>/<YEAR>/<TYPE>_<ID>`. The first call does not save anything.

The saved folder is then moved to `backup/`, outside `output/`: inside it, the build would read
it as one more event.

In [ ]:
S = "2 backup"
from sources.gdacs import api
from sources.gdacs.extract import event_dir_from_eventdata
from sources.utils.paths import parse_event_dir

EVENT_KEY = f"{EVENTTYPE}_{EVENTID}"
EVENT_DIR = None
with block(S, "event folder"):
    if RUN_DOWNLOADS:
        live = api.get_event_data(EVENTTYPE, EVENTID, save=False)
        EVENT_DIR = event_dir_from_eventdata(live, create=False)
        p = live["properties"]
        explain(f"GDACS {EVENT_KEY} (live answer, not saved)", name=p.get("name"), start=p.get("fromdate"),
                end=p.get("todate"), modified=p.get("datemodified"), country=p.get("iso3"),
                alert=f"{p.get('alertlevel')} (score {p.get('alertscore')})",
                severity=(p.get("severitydata") or {}).get("severitytext"),
                episodes=episodes_of(p), shakemap_versions=len(p.get("shakemap") or []), folder=EVENT_DIR.as_posix())
    else:
        found = sorted(Path("output").glob(f"*/*/{EVENT_KEY}/gdacs/eventdata.json"))
        if not found:
            raise RuntimeError(f"{EVENT_KEY} is not saved in output/ and RUN_DOWNLOADS is False")
        EVENT_DIR = found[0].parent.parent
    record(S, "event folder", "PASS", EVENT_DIR.as_posix())

INFO = parse_event_dir(EVENT_DIR)
ISO3, YEAR = INFO["iso3"], INFO["year"]
GD = EVENT_DIR / "gdacs"
COUNTRY_DIR = Path("output") / ISO3 / str(YEAR) / "country"
DATASET_DIR = Path("dataset") / ISO3 / str(YEAR)

BACKUP = None
with block(S, "backup"):
    need_downloads()
    if not DO_BACKUP:
        raise Skip("DO_BACKUP is False: files already saved are kept, impact runs and ShakeMaps are not downloaded again")
    BACKUP = Path("backup") / f"{EVENT_KEY}_{datetime.now():%Y%m%d_%H%M%S}"
    moved = []
    if EVENT_DIR.exists():
        shutil.move(str(EVENT_DIR), str(BACKUP / "output" / EVENT_KEY))
        moved.append(f"{EVENT_DIR.as_posix()} -> {(BACKUP / 'output' / EVENT_KEY).as_posix()} (moved)")
    for src, name in ((DATASET_DIR, "dataset_before"), (COUNTRY_DIR, "country_before")):
        if src.exists():
            shutil.copytree(src, BACKUP / name)
            moved.append(f"{src.as_posix()} -> {(BACKUP / name).as_posix()} (copied)")
    for line in moved or ["nothing was saved before"]:
        print("   ", line)
    record(S, "backup", "PASS", BACKUP.as_posix())

# the last backup of this event, for the comparisons of sections 3c and 14g
BACKUPS = sorted(Path("backup").glob(f"{EVENT_KEY}_*"))
LAST_BACKUP = BACKUP or (BACKUPS[-1] if BACKUPS else None)
explain("Folders", event=EVENT_DIR.as_posix(), country=COUNTRY_DIR.as_posix(), dataset=DATASET_DIR.as_posix(),
        backup=LAST_BACKUP.as_posix() if LAST_BACKUP else "none")

## 3 - GDACS, one function at a time

Each function of `sources/gdacs/api.py` calls one endpoint and saves its answer as it is.

In [ ]:
S = "3 gdacs"
from sources.gdacs import geometry
from sources.gdacs.extract_variables import save_variables

with block(S, "event data"):
    need_downloads()
    ev = api.get_event_data(EVENTTYPE, EVENTID)
    check(S, "folder from the saved answer", event_dir_from_eventdata(ev, create=False) == EVENT_DIR, EVENT_DIR.as_posix())

for name, call in (("card", lambda: api.get_event_card(EVENTTYPE, EVENTID, EVENT_DIR)),
                   ("EM-DAT", lambda: api.get_emdat(EVENTTYPE, EVENTID, EVENT_DIR)),
                   ("EMM news", lambda: api.get_emm_news(EVENTTYPE, EVENTID, EVENT_DIR, limit=NEWS_LIMIT))):
    with block(S, name):
        need_downloads()
        data = call()
        n = len(data) if isinstance(data, (list, dict)) else "?"
        record(S, name, "PASS", f"{type(data).__name__} with {n} items")

P = load_json(GD / "eventdata.json")["properties"] if (GD / "eventdata.json").is_file() else {}
EPISODES = episodes_of(P)
MAIN_EPISODE = int(P["episodeid"]) if P.get("episodeid") is not None else None
explain("Episodes", main=MAIN_EPISODE, all=EPISODES)

for ep in EPISODES:
    with block(S, f"episode {ep}"):
        need_downloads()
        api.get_episode_data(EVENTTYPE, EVENTID, ep, EVENT_DIR)
        api.get_geometry(EVENTTYPE, EVENTID, ep, EVENT_DIR)
        areas = geometry.explode_geometry(EVENT_DIR / "episodes" / str(ep) / "geometry.json", verbose=False)
        res = api.get_episode_impacts(EVENT_DIR, ep)
        record(S, f"episode {ep}", "PASS", f"{len(areas)} areas, impact runs {res['saved']}")
        warn(S, f"episode {ep}: every impact run answered", not res["failed"], res["failed"] or "")

with block(S, "ShakeMap versions"):
    need_downloads()
    res = api.get_shakemap_versions(EVENT_DIR)
    warn(S, "ShakeMap files answered", not res["failed"], f"{len(res['saved'])} files" + (f", failed {res['failed']}" if res["failed"] else ""))

with block(S, "variables.csv"):
    path = save_variables(EVENT_DIR)        # reads the saved files, no network
    record(S, "variables.csv", "PASS", Path(path).as_posix())

In [ ]:
with block(S, "saved files"):
    for name in ("eventdata", "adaptivecard", "emdat", "emm_news"):
        saved(S, GD / f"{name}.json", f"{name}.json")
    for ep in EPISODES:
        ep_dir = EVENT_DIR / "episodes" / str(ep)
        saved(S, ep_dir / "episodedata.json", f"episode {ep} episodedata.json")
        saved(S, ep_dir / "geometry.json", f"episode {ep} geometry.json")
        props = load_json(ep_dir / "episodedata.json").get("properties") or {}
        wanted = sorted(f"{kind}_{m.group(1)}.json" for imp in props.get("impacts") or []
                        for kind, url in (imp.get("resource") or {}).items()
                        if (m := re.search(r"getimpact\?id=(\d+)", url or "")))
        have = sorted(f.name for f in (ep_dir / "impacts").glob("*.json"))
        warn(S, f"episode {ep}: impact runs listed and saved", set(wanted) <= set(have), f"listed {wanted}, saved {have}")
        rows = []
        for f in sorted((ep_dir / "areas").glob("*/area.geojson")):
            feat = load_json(f)["features"][0]
            geom = feat.get("geometry") or {}
            polys = [geom["coordinates"]] if geom.get("type") == "Polygon" else geom.get("coordinates") if geom.get("type") == "MultiPolygon" else []
            rows.append({"area": f.parent.name, "geometry": geom.get("type"), "intensity": (feat.get("properties") or {}).get("intensity"),
                         "polygons": len(polys), "holes": sum(len(p) - 1 for p in polys),
                         "ShakeMap": (feat.get("properties") or {}).get("shakeid")})
        print(f"\nAreas of episode {ep} (intensity bands are rings: Intensity_4.5 is the band from 4.5 to 5):")
        display(pd.DataFrame(rows))
    versions = sorted(P.get("shakemap") or [], key=lambda s: s.get("shakeid") or 0)
    for v in versions:
        folder = GD / "shakemaps" / str(v.get("shakeid"))
        have = sorted(f.stem for f in folder.glob("*.json"))
        warn(S, f"ShakeMap {v.get('shakeid')} files", {"details", "geometry", "impact"} <= set(have),
             f"{have}, inserted {v.get('dateinserted')}, last={v.get('last')}")
    saved(S, EVENT_DIR / "variables.csv", "variables.csv")

### 3b - The API now against the saved files

Every endpoint is called again without saving. A difference is a WARN: GDACS rewrites its
records, so it means the event changed after the download.

In [ ]:
S = "3b live API"
with block(S, "API now against the saved files"):
    if not CHECK_LIVE_API:
        raise Skip("CHECK_LIVE_API is False")
    pairs = [("eventdata", lambda: api.get_event_data(EVENTTYPE, EVENTID, save=False), GD / "eventdata.json"),
             ("card", lambda: api.get_event_card(EVENTTYPE, EVENTID, EVENT_DIR, save=False), GD / "adaptivecard.json"),
             ("EM-DAT", lambda: api.get_emdat(EVENTTYPE, EVENTID, EVENT_DIR, save=False), GD / "emdat.json"),
             ("EMM news", lambda: api.get_emm_news(EVENTTYPE, EVENTID, EVENT_DIR, limit=NEWS_LIMIT, save=False), GD / "emm_news.json")]
    for ep in EPISODES:
        ep_dir = EVENT_DIR / "episodes" / str(ep)
        pairs += [(f"episode {ep}", lambda ep=ep: api.get_episode_data(EVENTTYPE, EVENTID, ep, EVENT_DIR, save=False), ep_dir / "episodedata.json"),
                  (f"geometry {ep}", lambda ep=ep: api.get_geometry(EVENTTYPE, EVENTID, ep, EVENT_DIR, save=False), ep_dir / "geometry.json")]
        for f in sorted((ep_dir / "impacts").glob("*.json")):
            kind, _, iid = f.stem.rpartition("_")
            pairs.append((f"impact {f.stem}", lambda iid=iid, ep=ep, kind=kind: api.get_impact(iid, EVENT_DIR, ep, kind, save=False), f))
    for name, call, file in pairs:
        with block(S, name):
            if not file.is_file():
                raise Skip(f"{file.as_posix()} not saved")
            d = diff_paths(call(), load_json(file))
            warn(S, name, not d, "identical" if not d else f"{len(d)} differences, e.g. {d[:6]}")

### 3c - Old download against the new one

The fields that GDACS can rewrite, in the backup and in the new download.

In [ ]:
S = "3c old vs new"
with block(S, "old download against the new one"):
    old_dir = LAST_BACKUP / "output" / EVENT_KEY if LAST_BACKUP else None
    if old_dir is None or not (old_dir / "gdacs" / "eventdata.json").is_file():
        raise Skip("no backup of this event")

    def facts(d):
        p = load_json(d / "gdacs" / "eventdata.json")["properties"]
        em = load_json(d / "gdacs" / "emdat.json") if (d / "gdacs" / "emdat.json").is_file() else []
        em = (em[0] if isinstance(em, list) and em else em) or {}
        news = load_json(d / "gdacs" / "emm_news.json") if (d / "gdacs" / "emm_news.json").is_file() else []
        return {"datemodified": p.get("datemodified"), "alertlevel": p.get("alertlevel"), "alertscore": p.get("alertscore"),
                "severity": (p.get("severitydata") or {}).get("severity"), "episodes": episodes_of(p),
                "ShakeMap versions": len(p.get("shakemap") or []), "EM-DAT deaths": em.get("totaldeaths"),
                "EM-DAT affected": em.get("totalaffected"), "EM-DAT homeless": em.get("numhomeless"), "news": len(news),
                "impact files": sorted(f.name for f in d.glob("episodes/*/impacts/*.json")),
                "areas": sorted({f.parent.name for f in d.glob("episodes/*/areas/*/area.geojson")})}

    old, new = facts(old_dir), facts(EVENT_DIR)
    table = pd.DataFrame({"before": pd.Series(old, dtype=object), "now": pd.Series(new, dtype=object)})
    table["changed"] = [str(a) != str(b) for a, b in zip(table["before"], table["now"])]
    display(table)
    record(S, "old download against the new one", "PASS", f"{int(table['changed'].sum())} fields changed: {list(table.index[table['changed']])}")

### 3d - variables.csv against the JSON files

`variables.csv` is the first table built from the GDACS files. A few values are rebuilt here by hand;
section 14a then checks every value of the dataset.

In [ ]:
S = "3d variables.csv"
with block(S, "variables.csv"):
    V = pd.read_csv(EVENT_DIR / "variables.csv")
    ev_json = load_json(GD / "eventdata.json")
    lon, lat = ev_json["geometry"]["coordinates"][:2]

    def value(level, variable, group=None, entity=None):
        r = V[(V["level"] == level) & (V["variable"] == variable)]
        if group is not None:
            r = r[r["group"] == group]
        if entity is not None:
            r = r[r["entity_id"].astype(str) == str(entity)]
        return r["value"].iloc[0] if len(r) else None

    explain("Epicentre", geometry=f"lon {lon}, lat {lat}", note="GeoJSON writes [lon, lat]")
    same(S, "severity", value("event", "severity"), (P.get("severitydata") or {}).get("severity"))
    same(S, "longitude", value("event", "longitude", "identification"), lon)
    same(S, "latitude", value("event", "latitude", "identification"), lat)
    same(S, "number of episodes", value("event", "n_episodes"), len(EPISODES))
    for v in P.get("shakemap") or []:
        sid = f"shake_{v.get('shakeid')}"
        la, lo = value("shakemap_version", "latitude", entity=sid), value("shakemap_version", "longitude", entity=sid)
        explain(f"ShakeMap {v.get('shakeid')}", conditions=v.get("conditions"), read_as=f"lat {la}, lon {lo}")
        km = haversine_km(float(la), float(lo), lat, lon) if is_number(la) and is_number(lo) else None
        warn(S, f"ShakeMap {v.get('shakeid')}: lat/lon swapped back, near the epicentre", km is not None and km < 50,
             f"{km:.1f} km from the GDACS epicentre" if km is not None else "no coordinates")
    card = {r["key"]: r["value"] for r in (load_json(GD / "adaptivecard.json").get("items") or [{}])[0].get("summarytable", [])}
    text = card.get("Exposed population")
    for part in str(text or "").split(" - "):
        num = re.search(r"([\d][\d.,]*)\s*(thousand|million|billion)?", part, re.IGNORECASE)
        mmi = re.search(r"MMI\s*>=\s*([IVX]+)", part)
        if num and mmi:
            scale = {"thousand": 1e3, "million": 1e6, "billion": 1e9}.get((num.group(2) or "").lower(), 1)
            same(S, f"exposed population MMI>={mmi.group(1)} from '{part.strip()}'",
                 value("event", f"exposed_population_mmi_ge_{mmi.group(1)}"), round(float(num.group(1).replace(",", "")) * scale))

## 4 - Boundaries

Downloaded again from FieldMaps (OCHA COD). The province of the epicentre is used in section 6.

In [ ]:
S = "4 boundaries"
from sources.boundaries import available_levels, download_boundaries, load_admin

with block(S, "download"):
    need_downloads()
    levels = download_boundaries(ISO3)
    check(S, "download", len(levels) > 0, {k: len(v) for k, v in levels.items()})

PROVINCE = None
with block(S, "saved files"):
    found = available_levels(ISO3)
    if not check(S, "levels saved", 0 in found and len(found) >= 2, found):
        raise Skip("no boundaries")
    a1, a2 = load_admin(ISO3, 1), load_admin(ISO3, 2)
    check(S, "admin 1 Pcodes unique", a1["pcode"].is_unique, f"{len(a1)} units")
    check(S, "admin 2 Pcodes unique", a2["pcode"].is_unique, f"{len(a2)} units")
    check(S, "every admin 2 has its admin 1", set(a2["parent_pcode"]) <= set(a1["pcode"]))
    check(S, "CRS is EPSG:4326", a2.crs.to_epsg() == 4326)
    bad = int((~a2.geometry.is_valid).sum())
    warn(S, "valid geometries", bad == 0, f"{bad} invalid")

    from shapely.geometry import Point
    lon, lat = load_json(GD / "eventdata.json")["geometry"]["coordinates"][:2]
    point = Point(lon, lat)
    hit = a1[a1.contains(point)]
    how = "contains the epicentre"
    if hit.empty:                               # epicentre at sea: the closest province
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            hit = a1.loc[[a1.distance(point).idxmin()]]
        how = "closest to the epicentre (it is not inside any province)"
    PROVINCE = hit["pcode"].iloc[0]
    explain("Province of the epicentre", pcode=PROVINCE, name=hit["name"].iloc[0], how=how,
            admin2_units=int((a2["parent_pcode"] == PROVINCE).sum()))

## 5 - GHSL for every area of the event

For each polygon of the main episode: population (100 m) and settlement class (1 km) of every cell
that touches it. `Centroid` is a point and has no cells.

Each area is then checked **without exactextract**: rasterio marks every cell the polygon touches
(`all_touched`), and the sums must agree within `REL_RASTER`. Cells touched only along an edge
can make a small difference.

In [ ]:
S = "5 ghsl location"
GHSL_AREAS = {}


def rasters():
    if POP_TIF and SMOD_TIF:
        return SMOD_TIF, POP_TIF
    from sources.ghsl.ghsl import RELEASE_ROOT, get_smod_and_pop
    if not RELEASE_ROOT.is_dir():
        raise Skip(f"GHSL rasters not found in {RELEASE_ROOT}: set POP_TIF and SMOD_TIF")
    return get_smod_and_pop(YEAR)


def touching(geom, tif):
    """Values of all the cells touching a geometry, read with rasterio (no exactextract)."""
    import rasterio
    from rasterio.features import rasterize
    from rasterio.windows import from_bounds
    with rasterio.open(tif) as src:
        b = geom.bounds
        w = from_bounds(b[0] - 1000, b[1] - 1000, b[2] + 1000, b[3] + 1000, src.transform).round_offsets().round_lengths()
        data, tr, nodata = src.read(1, window=w), src.window_transform(w), src.nodata
    mask = rasterize([(geom, 1)], out_shape=data.shape, transform=tr, all_touched=True, fill=0).astype(bool)
    values = data[mask]
    return values[values != nodata] if nodata is not None else values


with block(S, "rasters"):
    if not RUN_GHSL:
        raise Skip("RUN_GHSL is False")
    from sources.ghsl.ghsl import ghsl_tif_info
    SMOD_FILE, POP_FILE = rasters()
    for tif in (POP_FILE, SMOD_FILE):
        i = ghsl_tif_info(tif)
        explain(f"GHSL {i['product']}", epoch=i["epoch"], resolution_m=i["resolution_m"], crs=i["crs"], file=tif)
    record(S, "rasters", "PASS", f"epoch {ghsl_tif_info(POP_FILE)['epoch']} for the year {YEAR}")

if RUN_GHSL and MAIN_EPISODE is not None:
    from sources import ghsl
    from sources.gdacs.geometry import list_areas
    from sources.utils.geo import dissolve_features, read_features, to_raster_crs
    for label, f in list_areas(EVENT_DIR / "episodes" / str(MAIN_EPISODE)).items():
        with block(S, label):
            kind = load_json(f)["features"][0]["geometry"]["type"]
            if kind not in ("Polygon", "MultiPolygon"):
                raise Skip(f"{kind}: GHSL needs an area")
            SMOD_FILE, POP_FILE = rasters()
            gdf = to_raster_crs(dissolve_features(read_features(f)), POP_FILE)
            pop_cells, pop_area = ghsl.extract_pop(gdf, POP_FILE, f.parent)
            smod_cells, smod_area = ghsl.extract_smod(gdf, SMOD_FILE, f.parent)
            geom = gdf.geometry.iloc[0]
            pop = touching(geom, POP_FILE)
            cells = np.round(touching(geom, SMOD_FILE)).astype(int)
            explain(f"{label}", cells_100m=len(pop_cells), people=f"{pop_cells['pop'].sum():,.0f}",
                    people_weighted=f"{pop_area['pop_sum'].sum():,.0f}", cells_1km=len(smod_cells),
                    majority_class=int(smod_area["smod_majority"].iloc[0]))
            same(S, f"{label}: people in the touched cells, rasterio", pop_cells["pop"].sum(), float(pop[pop > 0].sum()), rel=REL_RASTER)
            same(S, f"{label}: 1 km cells, rasterio", len(smod_cells), len(cells), rel=REL_RASTER)
            ours = smod_cells["smod"].astype(int).value_counts().sort_index()
            theirs = pd.Series(cells).value_counts().sort_index()
            gap = (ours.reindex(theirs.index.union(ours.index), fill_value=0) - theirs.reindex(theirs.index.union(ours.index), fill_value=0)).abs().sum()
            check(S, f"{label}: 1 km cells by class, rasterio", gap <= max(2, REL_RASTER * len(cells)), f"{int(gap)} cells differ: {ours.to_dict()}")
            same(S, f"{label}: weighted people, cells against area table", pop_cells["pop_covered"].sum(), pop_area["pop_sum"].sum(), rel=1e-3)
            cov = pop_cells["coverage"]
            check(S, f"{label}: coverage in (0, 1]", bool(((cov > 0) & (cov <= 1 + 1e-9)).all()), f"{cov.min():.4f} .. {cov.max():.4f}")
            GHSL_AREAS[label] = float(pop_area["pop_sum"].sum())
else:
    record(S, "areas", "SKIP", "RUN_GHSL is False or no main episode")

## 6 - GHSL by admin unit

By default only the admin 2 units of the province of the epicentre, **not saved**: a partial run
saved in `country/ghsl/` would give admin 0 the total of one province. With `RUN_GHSL_FULL = True`
the whole country is computed and saved, and section 14 checks it.

Population of a unit = people in all the cells that touch it. A cell on a border counts for both
neighbours, so admin 2 adds up to more than admin 1.

In [ ]:
S = "6 ghsl admin"
GHSL_ADMIN = None
with block(S, "calculation"):
    if not RUN_GHSL:
        raise Skip("RUN_GHSL is False")
    if PROVINCE is None:
        raise Skip("no province (section 4)")
    from sources.ghsl.admin import admin_stats
    SMOD_FILE, POP_FILE = rasters()
    a2 = load_admin(ISO3, 2)
    if RUN_GHSL_FULL:
        GHSL_ADMIN = admin_stats(ISO3, YEAR, pop_tif=POP_FILE, smod_tif=SMOD_FILE)
        units = sorted(a2["pcode"])
    else:
        units = sorted(a2[a2["parent_pcode"] == PROVINCE]["pcode"])
        GHSL_ADMIN = admin_stats(ISO3, YEAR, pop_tif=POP_FILE, smod_tif=SMOD_FILE, pcodes=units, save=False)
    t1, t2 = GHSL_ADMIN[1], GHSL_ADMIN[2]
    check(S, "one row per admin 2 unit", len(t2) == len(units) and t2["pcode"].is_unique, f"{len(t2)} rows, {len(units)} units")
    check(S, "border cells count twice: admin 2 >= admin 1", t2["pop_ghsl"].sum() >= t1["pop_ghsl"].sum() - 1e-6,
          f"admin 2 {t2['pop_ghsl'].sum():,.0f}, admin 1 {t1['pop_ghsl'].sum():,.0f}")
    classes = [c for c in t2.columns if re.fullmatch(r"pop_ghsl_\d+", c)]
    gap = (t2["pop_ghsl"] - t2[classes].sum(axis=1)).max()
    warn(S, "people by class add up to the total", gap <= 1e-3 * max(1.0, t2["pop_ghsl"].max()),
         f"largest gap {gap:,.1f} people (people whose 1 km cell was not found)")

with block(S, "independent check"):
    if GHSL_ADMIN is None:
        raise Skip("no calculation")
    SMOD_FILE, POP_FILE = rasters()
    units = load_admin(ISO3, 2).to_crs("ESRI:54009").set_index("pcode")
    for _, r in GHSL_ADMIN[2].head(3).iterrows():
        pop = touching(units.loc[r["pcode"]].geometry, POP_FILE)
        cells = touching(units.loc[r["pcode"]].geometry, SMOD_FILE)
        same(S, f"people of {r['pcode']}, rasterio", r["pop_ghsl"], float(pop[pop > 0].sum()), rel=REL_RASTER)
        same(S, f"1 km cells of {r['pcode']}, rasterio", r.filter(like="smod_cells").sum(), len(cells), rel=REL_RASTER)

## 7 - IDMC

GIDD (validated, annual) and IDU (provisional). The IDU API serves the whole world in one file,
cached in `output/_cache/idmc/`.

The link between the event folder and an IDMC event is then **recomputed here**: same GLIDE, or same
country and hazard, start within 3 days and a figure within 300 km of the epicentre.

In [ ]:
S = "7 idmc"
with block(S, "download"):
    need_downloads()
    need_key("IDMC_CLIENT_ID")
    from sources.idmc.gidd import download_gidd_country
    from sources.idmc.numbers import list_events
    g = download_gidd_country(ISO3, YEAR, YEAR)
    record(S, "GIDD country totals", "PASS" if len(g) else "SKIP", f"{len(g)} rows" if len(g) else f"GIDD has no {YEAR} yet")
    ov = list_events(ISO3, YEAR, YEAR, top=10)
    events = ov["events"]
    record(S, "IDMC events", "PASS" if len(events) else "SKIP", f"{len(events)} events in {ISO3} {YEAR}")
    if len(events):
        quakes = events[events["hazard_or_violence_type"].astype(str).str.contains("Earthquake", na=False)]
        print(f"\nEarthquakes of {ISO3} in {YEAR} according to IDMC:")
        display(quakes[["event_id", "event_name", "event_start_date", "event_codes", "idu_flows", "gidd_flows", "in_idu", "in_gidd"]])

IDU = pd.DataFrame()
IDMC_LINKED, IDMC_GLIDES = [], []
GLIDE_RE = re.compile(r"[A-Z]{2}-\d{4}-\d{6}")
with block(S, "link with the event, recomputed"):
    cache = Path("output") / "_cache" / "idmc" / "idu_all.json"
    if not cache.is_file():
        raise Skip(f"{cache.as_posix()} not found: run the IDMC download")
    IDU = pd.DataFrame(load_json(cache))
    ev_json = load_json(GD / "eventdata.json")
    lon0, lat0 = ev_json["geometry"]["coordinates"][:2]
    start = pd.Timestamp(P["fromdate"])
    emdat = load_json(GD / "emdat.json") if (GD / "emdat.json").is_file() else []
    emdat = emdat if isinstance(emdat, list) else [emdat]
    own = set(GLIDE_RE.findall(str(P.get("glide") or "")) + GLIDE_RE.findall("|".join(str(r.get("externalids") or "") for r in emdat if r)))
    hazard = {"EQ": "Earthquake", "FL": "Flood", "TC": "Storm", "VO": "Volcanic activity", "WF": "Wildfire", "DR": "Drought"}[EVENTTYPE]
    countries = [c.get("iso3") for c in P.get("affectedcountries") or []] or [ISO3]
    cand = IDU[IDU["iso3"].isin(countries) & (IDU["type"] == hazard)]
    rows = []
    for ev_id, g in cand.groupby("event_id"):
        glides = {GLIDE_RE.search(c).group() for codes, types in zip(g["event_codes"], g["event_code_types"])
                  for c, t in zip(str(codes or "").split(";"), str(types or "").split(";"))
                  if "glide" in t.lower() and GLIDE_RE.search(c)}
        dt = (pd.Timestamp(g["event_start_date"].iloc[0]) - start).total_seconds() / 3600
        km = [haversine_km(lat0, lon0, la, lo) for la, lo in zip(g["latitude"], g["longitude"]) if is_number(la) and is_number(lo)]
        dist = min(km) if km else None
        why = "same GLIDE" if glides & own else ("within 3 days and 300 km" if abs(dt) <= 72 and dist is not None and dist <= 300 else "")
        rows.append({"idmc_event": str(ev_id), "name": g["event_name"].iloc[0], "start": g["event_start_date"].iloc[0],
                     "hours_from_gdacs": round(dt, 1), "km_from_epicentre": None if dist is None else round(dist, 1),
                     "glides": "|".join(sorted(glides)), "figures": len(g), "linked": why})
    table = pd.DataFrame(rows)
    if len(table):
        table = table.assign(_dt=table["hours_from_gdacs"].abs()).sort_values("_dt").drop(columns="_dt")
        print(f"\nIDU {hazard} events of {countries} (closest in time first), GDACS glides {sorted(own) or 'none'}:")
        display(table.head(15))
        IDMC_LINKED = sorted(table.loc[table["linked"] != "", "idmc_event"])
        IDMC_GLIDES = sorted({g for gl in table.loc[table["linked"] != "", "glides"] for g in gl.split("|") if g})
    from sources.dataset.idmc import link_event
    theirs = link_event(EVENT_DIR, IDU)
    theirs = sorted(theirs.loc[theirs["source"] == "idmc", "source_event_id"].astype(str))
    check(S, "same IDMC events as sources.dataset.idmc.link_event", theirs == IDMC_LINKED, f"recomputed {IDMC_LINKED}, pipeline {theirs}")
    explain("IDMC link", linked=IDMC_LINKED or "none: displacement of this event stays missing", glides=IDMC_GLIDES or "none")

## 8 - IOM DTM

In [ ]:
S = "8 iom"
with block(S, "download"):
    need_downloads()
    need_key("DTMAPI_SUBSCRIPTION_KEY")
    from sources.iom.iom import download_dtm
    res = download_dtm(ISO3, f"{YEAR}-01-01", f"{YEAR}-12-31", admin_levels=[0, 1, 2])
    n = {f"admin{k}": len(v) for k, v in res.items()}
    if not sum(n.values()):
        raise Skip(f"DTM has no {ISO3} data in {YEAR} {n}: normal for a country without a DTM operation")
    record(S, "download", "PASS", n)

## 9 - HDX HAPI

In [ ]:
S = "9 hdx"
with block(S, "download"):
    need_downloads()
    if not (os.getenv("HDX_APP_IDENTIFIER") or (os.getenv("HDX_APP_NAME") and os.getenv("HDX_APP_EMAIL"))):
        raise Skip("missing in .env: HDX_APP_NAME and HDX_APP_EMAIL")
    import sources.hdx as hdx
    for name in ("idps", "humanitarian_needs", "refugees", "returnees", "food_security", "food_prices", "poverty_rate"):
        with block(S, name):
            df = getattr(hdx, f"download_{name}")(ISO3, f"{YEAR}-01-01", f"{YEAR}-12-31", verbose=False)
            window = df.attrs.get("resolved_window")
            if not len(df):
                record(S, name, "SKIP", f"HAPI has no {name} for {ISO3}: {window}")
                continue
            warn(S, name, not str(window).startswith("fallback"), f"{len(df)} rows, {window}")

## 10 - INFORM

In [ ]:
S = "10 inform"
with block(S, "download"):
    need_downloads()
    from sources.inform import build_country_table
    t = build_country_table(ISO3, YEAR)
    explain("INFORM", edition=t["edition"].iloc[0], data_year=t["data_year"].iloc[0],
            matched=f"{int(t['matched'].sum())} of {len(t)}", validity_years=sorted(t["validity_year"].dropna().astype(int).unique()))
    check(S, "indicators found", int(t["matched"].sum()) >= 100, f"{int(t['matched'].sum())} of {len(t)}")
    check(S, "composite score found", bool(t.loc[t["indicator_id"] == "INFORM", "value"].notna().all()))

## 11 - ReliefWeb

ReliefWeb needs a GLIDE. GDACS and EM-DAT often have none: the notebook then tries the GLIDE of the
linked IDMC event, then `GLIDE` of section 0. A disaster that does not match the event (MISMATCH)
is moved to `reliefweb_rejected/`, so it stays out of the dataset.

In [ ]:
S = "11 reliefweb"
with block(S, "reports"):
    need_downloads()
    from sources.reliefweb import scrape_reliefweb
    from sources.reliefweb.reliefweb import event_from_variables
    tried = list(dict.fromkeys([g for g in [GLIDE] if g] + event_from_variables(EVENT_DIR)["glides"] + IDMC_GLIDES))
    if not tried:
        raise Skip("no GLIDE: GDACS and EM-DAT have none and no linked IDMC event has one. Set GLIDE in section 0 to try one")
    result = None
    for g in tried:
        result = scrape_reliefweb(g, EVENT_DIR, days_after=RELIEFWEB_DAYS)
        if result["verdict"] in ("MATCH", "PROBABLE"):
            break
    verdict = result["verdict"]
    if verdict in ("MATCH", "PROBABLE"):
        record(S, "reports", "PASS", f"{verdict} with {result['glide']}: {len(result['reports'])} reports")
    else:
        rejected = EVENT_DIR / "reliefweb_rejected"
        if rejected.exists():
            shutil.rmtree(rejected)
        shutil.move(str(EVENT_DIR / "reliefweb"), str(rejected))
        warn(S, "reports", False, f"{verdict} for {tried}: moved to {rejected.as_posix()}, out of the dataset")

## 12 - Montandon

In [ ]:
S = "12 montandon"
with block(S, "event"):
    if not RUN_MONTANDON:
        raise Skip("RUN_MONTANDON is False")
    need_downloads()
    need_key("MONTANDON_API_TOKEN")
    from sources import montandon
    client = montandon.connect()
    files = montandon.fetch_event(client, EVENTID, EVENT_DIR, EVENTTYPE)
    if not files:
        raise Skip("the event is not in Montandon")
    record(S, "event", "PASS", {k: Path(v).name for k, v in files.items()})

with block(S, "saved tables"):
    folder = EVENT_DIR / "montandon"
    if not folder.is_dir():
        raise Skip("nothing saved")
    for name in ("events", "impacts", "hazards"):
        f = folder / f"{name}.parquet"
        if f.is_file():
            t = pd.read_parquet(f)
            cols = [c for c in ("collection", "match", "kind") if c in t]
            print(f"\n{name}: {len(t)} rows")
            if cols:
                display(t.groupby(cols).size().rename("rows").reset_index())
    record(S, "saved tables", "PASS", sorted(p.name for p in folder.glob("*.parquet")))

## 13 - Build the dataset

Reads `output/` without network and rewrites the long tables and the tables of the country.
`dataset_variables.xlsx` must be closed.

In [ ]:
S = "13 build"
PROBLEMS = None
with block(S, "build"):
    from sources.dataset.build import build
    PROBLEMS = build(iso3=[ISO3])
    for p in PROBLEMS:
        print("   problem:", p)
    check(S, "no problems", not PROBLEMS, f"{len(PROBLEMS)} problems")

## 14 - Are the numbers right?

From here on nothing is downloaded or written. The dataset is read with `sources.dataset.access`,
the original files with `json` and `pandas`.

In [ ]:
S = "14 load"
from sources.dataset import access

EVENT_LEVELS = ("event", "episode", "location")
ADMIN_LEVELS = ("admin0", "admin1", "admin2")
LONG, T = {}, {}
for lv in (*ADMIN_LEVELS, *EVENT_LEVELS):
    try:
        LONG[lv] = access.load(lv)
    except FileNotFoundError:
        LONG[lv] = pd.DataFrame()
    try:
        T[lv] = access.table(ISO3, YEAR, lv)
    except FileNotFoundError:
        pass
COLS = access.columns(ISO3, YEAR) if (DATASET_DIR / "columns.csv").is_file() else pd.DataFrame()


def rows(level, source=None):
    """Long rows of this event (event levels) or of this country (admin levels)."""
    df = LONG.get(level, pd.DataFrame())
    if df.empty:
        return df
    df = df[df["event_id"] == EVENT_KEY] if level in EVENT_LEVELS else df[df["iso3"] == ISO3]
    return df[df["source"] == source] if source else df


# the original files read again here, without the extraction code
EV_JSON = load_json(GD / "eventdata.json")
P = EV_JSON["properties"]
EPISODES = episodes_of(P)
MAIN_EPISODE = int(P["episodeid"]) if P.get("episodeid") is not None else None
if IDU.empty and (Path("output") / "_cache" / "idmc" / "idu_all.json").is_file():
    IDU = pd.DataFrame(load_json(Path("output") / "_cache" / "idmc" / "idu_all.json"))

found = [rows(lv).groupby("source").size().rename("values").reset_index().assign(level=lv) for lv in LONG if len(rows(lv))]
if found:
    print(f"Values of {EVENT_KEY} and of {ISO3} in the long tables:")
    display(pd.concat(found, ignore_index=True)[["level", "source", "values"]])
for lv, t in T.items():
    print(f"table {lv}: {t.shape[0]} rows x {t.shape[1]} columns")
record(S, "tables", "PASS" if T else "FAIL", f"{len(T)} tables in {DATASET_DIR.as_posix()}")

### 14a - Original files against the long tables

For each source the values are rebuilt from the saved answers and compared one by one with the
long tables in `dataset/_long/`, together with their dates (`known_at`, `valid_from`, `valid_to`)
where the rule is known.

In [ ]:
S = "14a original -> long"


def card_of(event_dir):
    f = event_dir / "gdacs" / "adaptivecard.json"
    if not f.is_file():
        return {}
    item = (load_json(f).get("items") or [{}])[0]
    return {r.get("key"): r.get("value") for r in item.get("summarytable", [])}


def exposed(text):
    """'130 thousand (in MMI>=VII) - About 700 people in MMI>=IX' -> {'VII': 130000, 'IX': 700}"""
    out = {}
    for part in str(text or "").split(" - "):
        num = re.search(r"([\d][\d.,]*)\s*(thousand|million|billion)?", part, re.IGNORECASE)
        mmi = re.search(r"MMI\s*>=\s*([IVX]+)", part)
        if num and mmi:
            scale = {"thousand": 1e3, "million": 1e6, "billion": 1e9}.get((num.group(2) or "").lower(), 1)
            out[mmi.group(1).upper()] = round(float(num.group(1).replace(",", "")) * scale)
    return out


def scalars_of(data):
    """Scalars of an impact run, first value of each name, and the names given twice with another value."""
    out, conflicts = {}, []
    for group in (data or {}).get("datums") or []:
        for d in group.get("datum") or []:
            for s in ((d.get("scalars") or {}).get("scalar") or []):
                name = s.get("name")
                if name is None:
                    continue
                if name in out:
                    if str(out[name]) != str(s.get("value")):
                        conflicts.append(name)
                    continue
                out[name] = s.get("value")
    return out, conflicts


IMPACT_NAMES = {"PopulationOverVII": "impact_model_population_over_vii", "MaxMMI": "impact_model_max_mmi",
                "PopulationMaxIntensity": "impact_model_population_max_intensity", "CountryVuln": "impact_model_country_vuln",
                "ScaledPopulation": "impact_model_scaled_population", "AlertScore": "impact_model_alert_score",
                "LccNormalised": "impact_model_lcc_normalised", "CoeffIntersept": "impact_model_coeff_intercept",
                "CoeffSlope": "impact_model_coeff_slope"}


def impact_variable(name):
    if name.startswith("SUMPOP"):
        return "impact_model_sumpop"
    if re.fullmatch(r"ep\d+", name):
        return "impact_model_ep_mmi"
    return IMPACT_NAMES.get(name, f"impact_model_{name.lower()}")


def run_time(run):
    """'202508181603471257' -> '2025-08-18T16:03:47'"""
    s = str(run or "")
    return f"{s[:4]}-{s[4:6]}-{s[6:8]}T{s[8:10]}:{s[10:12]}:{s[12:14]}" if len(s) >= 14 and s[:14].isdigit() else None


from pyproj import Geod
from shapely.geometry import shape

SPHERE = Geod(a=6378137, b=6378137)
WGS84 = Geod(ellps="WGS84")


def area_km2(geom, geod=SPHERE):
    if not geom or geom.get("type") not in ("Polygon", "MultiPolygon"):
        return None
    return abs(geod.geometry_area_perimeter(shape(geom))[0]) / 1e6


SHAKE_DATE = {str(s.get("shakeid")): s.get("dateinserted") for s in P.get("shakemap") or []}
WHEN = dict(known_at=P.get("datemodified"), valid_from=P.get("fromdate"), valid_to=P.get("todate"))

with block(S, "GDACS event"):
    lon, lat = EV_JSON["geometry"]["coordinates"][:2]
    sev = P.get("severitydata") or {}
    card = card_of(EVENT_DIR)
    ex = exposed(card.get("Exposed population"))
    local = card.get("Local Time")
    local = datetime.strptime(local.strip(), "%d %b %Y %H:%M").isoformat() if local else None
    items = {  # variable: (number, text)
        "event_gdacs_id": (None, as_text(P.get("eventid"))), "event_name": (None, as_text(P.get("name"))),
        "event_glide": (None, as_text(P.get("glide"))), "event_country_name": (None, as_text(P.get("country"))),
        "event_affected_iso3": (None, "|".join(str(c.get("iso3")) for c in P.get("affectedcountries") or []) or None),
        "event_n_episodes": (len(EPISODES), None), "event_main_episode_id": (None, as_text(P.get("episodeid"))),
        "event_report_url": (None, as_text((P.get("url") or {}).get("report"))), "event_usgs_id": (None, as_text(P.get("sourceid"))),
        "event_hazard_agency": (None, as_text(P.get("source"))), "event_local_time": (None, local),
        "hazard_severity": (sev.get("severity"), None), "hazard_severity_text": (None, as_text(sev.get("severitytext"))),
        "hazard_lat": (lat, None), "hazard_lon": (lon, None),
        "hazard_magnitude": (first_number(card.get("Magnitude")), None), "hazard_depth": (first_number(card.get("Depth")), None),
        "hazard_tsunami_wave_height": (first_number(card.get("Maximun Wave Height estimated")), None),
        "alert_level": (None, as_text(P.get("alertlevel"))), "alert_score": (P.get("alertscore"), None),
        "exposure_pop_mmi_ge_7": (ex.get("VII"), None), "exposure_pop_mmi_ge_9": (ex.get("IX"), None),
        "exposure_pop_text": (None, as_text(card.get("Exposed population"))),
    }
    explain("GDACS event", file=(GD / "eventdata.json").as_posix(), epicentre=f"lat {lat}, lon {lon}",
            card_magnitude=card.get("Magnitude"), card_depth=card.get("Depth"), exposure=card.get("Exposed population"))
    compare(S, "event", "gdacs", rows("event", "gdacs"),
            [dict(WHEN, variable=k, value=v, text=t) for k, (v, t) in items.items()], label="GDACS event")

with block(S, "ShakeMap versions"):
    exp = []
    for sm in P.get("shakemap") or []:
        c = {k.strip(): v for k, v in (kv.split("=", 1) for kv in str(sm.get("conditions") or "").split(";") if "=" in kv)}
        base = dict(source_version=str(sm.get("shakeid")), known_at=sm.get("dateinserted"), valid_from=P.get("fromdate"))
        exp += [dict(base, variable="shakemap_magnitude", value=c.get("magnitude")),
                dict(base, variable="shakemap_depth", value=c.get("depth")),
                dict(base, variable="shakemap_max_mmi", value=c.get("maxmmi")),
                dict(base, variable="shakemap_lat", value=c.get("longitude")),     # GDACS writes lat and lon swapped
                dict(base, variable="shakemap_lon", value=c.get("latitude")),
                dict(base, variable="shakemap_is_last", text=as_text(sm.get("last")))]
    for folder in sorted((GD / "shakemaps").glob("*")):
        if (folder / "details.json").is_file():
            d = load_json(folder / "details.json").get("properties") or {}
            base = dict(source_version=folder.name, known_at=d.get("dateadded"), valid_from=P.get("fromdate"))
            exp += [dict(base, variable="shakemap_alert_score", value=d.get("alertscore")),
                    dict(base, variable="shakemap_alert_level", text=d.get("alertelevel"))]
    compare(S, "event", "gdacs_shakemap", rows("event", "gdacs_shakemap"), exp, keys=["source_version"], label="ShakeMap versions")

with block(S, "ShakeMap impact"):
    exp = []
    for folder in sorted((GD / "shakemaps").glob("*")):
        f, det = folder / "impact.json", folder / "details.json"
        if not (f.is_file() and det.is_file()):
            continue
        values, conflicts = scalars_of(load_json(f))
        warn(S, f"ShakeMap {folder.name} impact: one value per name", not conflicts, conflicts or "")
        known = (load_json(det).get("properties") or {}).get("dateadded")
        exp += [dict(source_version=folder.name, population_group=n, variable=impact_variable(n), value=v, known_at=known)
                for n, v in values.items() if is_number(v)]
    compare(S, "event", "gdacs_shakemap_impact", rows("event", "gdacs_shakemap_impact"), exp,
            keys=["source_version", "population_group"], label="ShakeMap impact")

with block(S, "GDACS episodes"):
    exp = []
    for ep in EPISODES:
        f = EVENT_DIR / "episodes" / str(ep) / "episodedata.json"
        if not f.is_file():
            continue
        q = load_json(f).get("properties") or {}
        eq = q.get("earthquakedetails") or {}
        ts = eq.get("tsmaxheight")
        items = {"episode_alert_level": (None, as_text(q.get("episodealertlevel"))), "episode_alert_score": (q.get("episodealertscore"), None),
                 "alert_level": (None, as_text(q.get("alertlevel"))), "alert_score": (q.get("alertscore"), None),
                 "episode_is_main": (None, str(ep == MAIN_EPISODE).lower()),
                 "episode_is_temporary": (None, as_text(q.get("istemporary"))), "episode_is_current": (None, as_text(q.get("iscurrent"))),
                 "hazard_magnitude": (eq.get("magnitude"), None), "hazard_depth": (eq.get("depth"), None),
                 "hazard_tsunami_max_height": (first_number(ts) if ts not in (None, "") else None, None),
                 "exposure_pop_mmi_ge_7_shakemap": (eq.get("shakepop"), None), "exposure_pop_mmi_ge_7_rapid": (eq.get("rapidpop"), None)}
        exp += [dict(WHEN, episode_id=ep, variable=k, value=v, text=t) for k, (v, t) in items.items()]
    compare(S, "episode", "gdacs", rows("episode", "gdacs"), exp, keys=["episode_id"], label="GDACS episodes")

with block(S, "impact runs"):
    exp = []
    for f in sorted(EVENT_DIR.glob("episodes/*/impacts/*.json")):
        kind = f.stem.rpartition("_")[0]
        data = load_json(f)
        values, conflicts = scalars_of(data)
        warn(S, f"{f.name}: one value per name", not conflicts, conflicts or "")
        known = run_time(data.get("modelrun")) or P.get("datemodified")
        exp += [dict(episode_id=f.parent.parent.name, file=f.name, reason=kind, population_group=n, variable=impact_variable(n),
                     value=v, known_at=known) for n, v in values.items() if is_number(v)]
    D = rows("episode", "gdacs_impact")
    D = D.assign(file=D["source_file"].map(lambda s: Path(s).name)) if len(D) else D
    compare(S, "episode", "gdacs_impact", D, exp, keys=["episode_id", "file", "reason", "population_group"], label="impact runs")

with block(S, "GDACS areas"):
    exp = []
    for ep in EPISODES:
        ep_dir = EVENT_DIR / "episodes" / str(ep)
        if not (ep_dir / "geometry.json").is_file():
            continue
        q = load_json(ep_dir / "episodedata.json").get("properties") or {} if (ep_dir / "episodedata.json").is_file() else {}
        used = {}
        for i, feat in enumerate(load_json(ep_dir / "geometry.json").get("features") or []):
            pr, geom = feat.get("properties") or {}, feat.get("geometry") or {}
            label = label_of(pr.get("polygonlabel") or pr.get("Class") or f"feature_{i}")
            if label in used:
                used[label] += 1
                label = f"{label}_{used[label]}"
            else:
                used[label] = 0
            known = (SHAKE_DATE.get(str(pr.get("shakeid"))) if pr.get("shakeid") is not None else None) or pr.get("polygondate") or q.get("todate")
            area_file = ep_dir / "areas" / label / "area.geojson"
            n = len(geom.get("coordinates") or []) if geom.get("type") == "MultiPolygon" else 1
            items = {"location_label": (None, as_text(pr.get("polygonlabel"))), "location_class": (None, as_text(pr.get("Class"))),
                     "location_mmi": (pr.get("intensity"), None), "location_shakemap_id": (None, as_text(pr.get("shakeid"))),
                     "location_geometry_type": (None, as_text(geom.get("type"))), "location_n_polygons": (n, None),
                     "location_area_km2": (area_km2(geom), None),
                     "location_bbox": (None, "|".join(str(v) for v in feat["bbox"]) if feat.get("bbox") else None),
                     "location_geometry_file": (None, area_file.as_posix() if area_file.is_file() else None)}
            exp += [dict(episode_id=ep, location_id=label, variable=k, value=v, text=t, known_at=known, valid_from=P.get("fromdate"))
                    for k, (v, t) in items.items()]
    compare(S, "location", "gdacs", rows("location", "gdacs"), exp, keys=["episode_id", "location_id"],
            rels={"location_area_km2": REL_AREA}, label="GDACS areas")

with block(S, "ShakeMap bands"):
    exp = []
    for folder in sorted((GD / "shakemaps").glob("*")):
        f, det = folder / "geometry.json", folder / "details.json"
        if not (f.is_file() and det.is_file()):
            continue
        known = (load_json(det).get("properties") or {}).get("dateadded")
        for feat in load_json(f).get("features") or []:
            mmi = (feat.get("properties") or {}).get("intensity")
            if mmi is None:
                continue
            base = dict(source_version=folder.name, location_id=label_of(f"Intensity {float(mmi):g}"), known_at=known)
            exp += [dict(base, variable="location_mmi", value=mmi),
                    dict(base, variable="location_area_km2", value=area_km2(feat.get("geometry")))]
    compare(S, "location", "gdacs_shakemap", rows("location", "gdacs_shakemap"), exp, keys=["source_version", "location_id"],
            rels={"location_area_km2": REL_AREA}, label="ShakeMap bands")

In [ ]:
EMDAT_NUM = {"magnitude": "hazard_magnitude", "latitude": "hazard_lat", "longitude": "hazard_lon", "totaldeaths": "impact_deaths",
             "numinjured": "impact_injured", "numaffected": "impact_affected", "numhomeless": "impact_homeless",
             "totalaffected": "impact_total_affected", "totaldam": "impact_damage_total", "totaldamadj": "impact_damage_total_adjusted",
             "reconstrdam": "impact_damage_reconstruction", "reconstrdamadj": "impact_damage_reconstruction_adjusted",
             "insurdam": "impact_damage_insured", "insurdamadj": "impact_damage_insured_adjusted", "cpi": "impact_cpi",
             "aidcontribution": "response_aid_contribution"}
EMDAT_TEXT = {"disasternumber": "event_emdat_id", "classificationkey": "hazard_classification", "disastertype": "hazard_type_name",
              "disastersubtype": "hazard_subtype_name", "subregion": "event_region", "location": "impact_location_text",
              "ofdaresponse": "response_ofda", "appeal": "response_appeal", "declaration": "response_declaration"}

with block(S, "EM-DAT"):
    recs = load_json(GD / "emdat.json") if (GD / "emdat.json").is_file() else []
    recs = [r for r in (recs if isinstance(recs, list) else [recs]) if r]
    warn(S, "EM-DAT: one record", len(recs) <= 1, f"{len(recs)} records")
    exp = []
    for rec in recs:
        when = dict(valid_from=recs[0].get("fromDate"), valid_to=recs[0].get("toDate"))
        ext = "|".join(c.split(":", 1)[1] for c in str(rec.get("externalids") or "").split("|") if ":" in c) or None
        exp += [dict(when, variable=v, value=rec.get(k)) for k, v in EMDAT_NUM.items()]
        exp += [dict(when, variable=v, text=as_text(rec.get(k))) for k, v in EMDAT_TEXT.items()]
        exp.append(dict(when, variable="event_external_ids", text=ext))
        explain("EM-DAT record", id=rec.get("disasternumber"), deaths=rec.get("totaldeaths"), injured=rec.get("numinjured"),
                affected=rec.get("totalaffected"), homeless=rec.get("numhomeless"), glide=ext or "none")
    compare(S, "event", "emdat", rows("event", "emdat"), exp, label="EM-DAT")

with block(S, "EMM"):
    if not (GD / "emm_news.json").is_file():
        raise Skip("no emm_news.json")
    news = load_json(GD / "emm_news.json")
    dates = sorted(n.get("pubdate") for n in news if n.get("pubdate"))
    when = dict(valid_from=dates[0] if dates else None, valid_to=dates[-1] if dates else None, known_at=dates[-1] if dates else None)
    explain("EMM news", news=len(news), first=dates[0] if dates else None, last=dates[-1] if dates else None,
            note=f"only the latest {NEWS_LIMIT} are downloaded")
    compare(S, "event", "emm", rows("event", "emm"),
            [dict(when, variable="media_news_count", value=len(news)),
             dict(when, variable="media_source_count", value=len({n.get("source") for n in news}))], label="EMM")

In [ ]:
RURAL, URBAN, WATER = (11, 12, 13), (21, 22, 23, 30), 10
CLASSES = (10, 11, 12, 13, 21, 22, 23, 30)


def epoch_of(path):
    return int(re.search(r"_E(\d{4})[_.]", Path(path).name).group(1))


def epoch_dates(epoch):
    return dict(valid_from=f"{epoch}-01-01", valid_to=f"{epoch}-12-31")


with block(S, "GHSL areas"):
    exp = {"ghsl_pop": [], "ghsl_smod": [], "ghsl_pop_smod": []}
    aligned = []
    for area in sorted(EVENT_DIR.glob("episodes/*/areas/*")):
        base = dict(episode_id=area.parent.parent.name, location_id=area.name)
        for f in sorted(area.glob("100m/ghsl_pop_E*_area.parquet")):
            when = epoch_dates(epoch_of(f))
            exp["ghsl_pop"].append(dict(base, **when, variable="pop_ghsl_covered", value=pd.read_parquet(f)["pop_sum"].sum()))
            px = f.with_name(f.name.replace("_area", "_pixels"))
            if px.is_file():
                exp["ghsl_pop"].append(dict(base, **when, variable="pop_ghsl", value=pd.read_parquet(px, columns=["pop"])["pop"].sum()))
        for f in sorted(area.glob("1000m/ghsl_smod_E*_area.parquet")):
            epoch = epoch_of(f)
            when = epoch_dates(epoch)
            a = pd.read_parquet(f).iloc[0]
            frac = {int(c.split("_")[1]): float(a[c]) for c in a.index if c.startswith("frac_")}
            land = sum(v for k, v in frac.items() if k != WATER)
            exp["ghsl_smod"].append(dict(base, **when, variable="smod_majority_class", text=str(int(a["smod_majority"]))))
            exp["ghsl_smod"] += [dict(base, **when, variable="smod_area_share_class", population_group=str(k), value=v) for k, v in frac.items()]
            if land > 0:
                exp["ghsl_smod"] += [dict(base, **when, variable="smod_share_rural", value=sum(frac.get(c, 0) for c in RURAL) / land),
                                     dict(base, **when, variable="smod_share_urban", value=sum(frac.get(c, 0) for c in URBAN) / land)]
            pop_px = area / "100m" / f"ghsl_pop_E{epoch}_pixels.parquet"
            smod_px = f.with_name(f.name.replace("_area", "_pixels"))
            if not (pop_px.is_file() and smod_px.is_file()):
                continue
            sm = pd.read_parquet(smod_px, columns=["x", "y", "smod"])
            po = pd.read_parquet(pop_px, columns=["x", "y", "pop"])
            po = po[po["pop"] > 0]
            aligned.append(bool(((sm["x"].round() % 1000 == 500) & (sm["y"].round() % 1000 == 500)).all()))
            # each 100 m cell takes the class of the 1 km cell around its centre
            centre = lambda v: (np.floor(v / 1000) * 1000 + 500).round().astype("int64")
            classes = pd.DataFrame({"cx": sm["x"].round().astype("int64"), "cy": sm["y"].round().astype("int64"),
                                    "cls": sm["smod"].round().astype(int)}).drop_duplicates(["cx", "cy"])
            joined = pd.DataFrame({"cx": centre(po["x"]).to_numpy(), "cy": centre(po["y"]).to_numpy(), "pop": po["pop"].to_numpy()})
            joined = joined.merge(classes, on=["cx", "cy"], how="left")
            joined["cls"] = joined["cls"].fillna(-1).astype(int)
            by = joined.groupby("cls")["pop"].sum()
            total = by.sum()
            if total <= 0:
                continue
            rural, urban = float(by.reindex(RURAL, fill_value=0).sum()), float(by.reindex(URBAN, fill_value=0).sum())
            counts = sm["smod"].round().astype(int).value_counts()
            e = exp["ghsl_pop_smod"]
            e += [dict(base, **when, variable="pop_ghsl_class", population_group=str(k), value=v) for k, v in by.items() if k != -1]
            e += [dict(base, **when, variable="pop_ghsl_rural", value=rural), dict(base, **when, variable="pop_ghsl_urban", value=urban),
                  dict(base, **when, variable="pop_share_rural", value=rural / total)]
            e += [dict(base, **when, variable="smod_cells_class", population_group=str(k), value=n) for k, n in counts.items()]
            e += [dict(base, **when, variable="smod_cells_rural", value=int(counts.reindex(RURAL, fill_value=0).sum())),
                  dict(base, **when, variable="smod_cells_urban", value=int(counts.reindex(URBAN, fill_value=0).sum()))]
    if aligned:
        check(S, "GHSL: 1 km cell centres on x00 500 m (assumed by the POP x SMOD join)", all(aligned), f"{sum(aligned)} of {len(aligned)} areas")
    for src, e in exp.items():
        with block(S, src):
            compare(S, "location", src, rows("location", src), e, keys=["episode_id", "location_id", "population_group"], label=f"{src} areas")

with block(S, "GHSL admin"):
    files = {n: sorted((COUNTRY_DIR / "ghsl" / f"admin{n}").glob("ghsl_E*.parquet")) for n in (0, 1, 2)}
    if not any(files.values()):
        raise Skip("no GHSL admin tables: they are saved only with RUN_GHSL_FULL = True")
    for n, fl in files.items():
        exp = {"ghsl_pop": [], "ghsl_smod": [], "ghsl_pop_smod": []}
        for f in fl:
            when = epoch_dates(epoch_of(f))
            for _, u in pd.read_parquet(f).iterrows():
                base = dict(when, unit=u["pcode"] if n else ISO3)
                cells = {c: int(u[f"smod_cells_{c}"]) for c in CLASSES}
                people = {c: float(u[f"pop_ghsl_{c}"]) for c in CLASSES}
                land = sum(v for c, v in cells.items() if c != WATER)
                rural, urban = sum(people[c] for c in RURAL), sum(people[c] for c in URBAN)
                exp["ghsl_pop"].append(dict(base, variable="pop_ghsl", value=float(u["pop_ghsl"])))
                exp["ghsl_pop_smod"] += [dict(base, variable="pop_ghsl_class", population_group=str(c), value=v) for c, v in people.items()]
                exp["ghsl_pop_smod"] += [dict(base, variable="pop_ghsl_rural", value=rural), dict(base, variable="pop_ghsl_urban", value=urban)]
                if sum(people.values()) > 0:
                    exp["ghsl_pop_smod"].append(dict(base, variable="pop_share_rural", value=rural / sum(people.values())))
                exp["ghsl_smod"] += [dict(base, variable="smod_cells_class", population_group=str(c), value=v) for c, v in cells.items()]
                exp["ghsl_smod"] += [dict(base, variable="smod_cells_rural", value=sum(cells[c] for c in RURAL)),
                                     dict(base, variable="smod_cells_urban", value=sum(cells[c] for c in URBAN))]
                if land:
                    exp["ghsl_smod"] += [dict(base, variable="smod_share_rural", value=sum(cells[c] for c in RURAL) / land),
                                         dict(base, variable="smod_share_urban", value=sum(cells[c] for c in URBAN) / land)]
        for src, e in exp.items():
            with block(S, f"{src} admin{n}"):
                D = rows(f"admin{n}", src)
                D = D.assign(unit=D[f"admin{n}_pcode"] if n else D["iso3"]) if len(D) else D
                compare(S, f"admin{n}", src, D, e, keys=["unit", "population_group"], label=f"{src} admin{n}")

with block(S, "INFORM"):
    files = sorted(COUNTRY_DIR.glob("inform_*.csv"))
    if not files:
        raise Skip("no INFORM table")
    exp = []
    for f in files:
        t = pd.read_csv(f)
        t = t[t["matched"].astype(bool) & t["value"].notna()]
        for r in t.to_dict("records"):
            if is_number(r.get("validity_year")):
                y0 = y1 = int(float(r["validity_year"]))
            else:
                years = [int(y) for y in re.findall(r"\d{4}", str(r.get("reference_year")))] if not missing(r.get("reference_year")) else []
                years = years or [int(r["data_year"])]
                y0, y1 = min(years), max(years)
            exp.append(dict(file=f.name, variable="inform_" + slug(r["indicator_name"]), value=r["value"],
                            valid_from=f"{y0}-01-01", valid_to=f"{y1}-12-31", known_at=f"{int(r['edition'])}-01-01"))
    D = rows("admin0", "inform")
    D = D.assign(file=D["source_file"].map(lambda s: Path(s).name)) if len(D) else D
    compare(S, "admin0", "inform", D, exp, keys=["file"], label="INFORM")

In [ ]:
with block(S, "IOM DTM"):
    done = 0
    for n in (0, 1, 2):
        files = [f for f in sorted(COUNTRY_DIR.glob(f"iom/dtm_admin{n}_*.json")) if not f.name.endswith(".meta.json")]
        raw = pd.concat([pd.DataFrame(load_json(f)) for f in files], ignore_index=True) if files else pd.DataFrame()
        if raw.empty:
            continue
        done += 1
        if "id" in raw:
            raw = raw.drop_duplicates("id")
        raw["unit"] = raw[f"admin{n}Pcode"] if n else ISO3
        raw["date"] = pd.to_datetime(raw["reportingDate"]).dt.strftime("%Y-%m-%d")
        sums = pd.to_numeric(raw["numPresentIdpInd"], errors="coerce").groupby([raw["unit"], raw["date"]]).sum(min_count=1)
        exp = [dict(unit=u, date=d, variable="idp_stock", value=v) for (u, d), v in sums.items()]
        D = rows(f"admin{n}", "iom_dtm")
        D = D[D["gender"] == "all"] if len(D) else D
        if len(D):
            D = (D.assign(unit=D[f"admin{n}_pcode"] if n else D["iso3"], date=D["valid_from"].dt.strftime("%Y-%m-%d"))
                 .groupby(["unit", "date", "variable"], as_index=False)["value"].sum(min_count=1))
        compare(S, f"admin{n}", "iom_dtm", D, exp, keys=["unit", "date"], label=f"DTM admin{n} (sum by unit and date)")
    if not done:
        raise Skip("no DTM data saved for this country")

HDX_SETS = {  # folder: (source, {original column: dataset variables})
    "humanitarian_needs": ("hdx_needs", {"population": ("pop_hno", "needs_people")}),
    "idps": ("hdx_idps", {"population": ("idp_stock",)}),
    "food_prices": ("hdx_food_prices", {"price": ("food_price",)}),
    "food_security": ("hdx_food_security", {"population_in_phase": ("ipc_population",),
                                            "population_fraction_in_phase": ("ipc_population_share",)}),
    "poverty_rate": ("hdx_poverty", {"mpi": ("poverty_mpi",), "headcount_ratio": ("poverty_headcount_ratio",),
                                     "intensity_of_deprivation": ("poverty_intensity",), "vulnerable_to_poverty": ("poverty_vulnerable",),
                                     "in_severe_poverty": ("poverty_severe",)}),
    "refugees": ("hdx_refugees", {"population": ("refugee_population",)}),
    "returnees": ("hdx_returnees", {"population": ("returnee_population",)}),
}
for folder, (source, columns_) in HDX_SETS.items():
    with block(S, f"HDX {folder}"):
        files = sorted((COUNTRY_DIR / "hdx" / folder).glob(f"{folder}_*.parquet"))
        raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
        if raw.empty:
            raise Skip("nothing saved for this country")
        raw = raw.loc[raw.astype(str).drop_duplicates().index]          # the same row from two downloads
        level = raw["admin_level"] if "admin_level" in raw else pd.Series(0, index=raw.index)
        for n in sorted(level.dropna().astype(int).unique()):
            part = raw[level == n]
            unit = part[f"admin{n}_code"] if n else pd.Series(ISO3, index=part.index)
            date = part["reference_period_start"].astype(str).str[:10]
            for col, variables in columns_.items():
                sums = pd.to_numeric(part[col], errors="coerce").groupby([unit.values, date.values]).sum(min_count=1)
                exp = [dict(unit=u, date=d, variable=col, value=v) for (u, d), v in sums.items()]
                D = rows(f"admin{n}", source)
                D = D[D["variable"].isin(variables)] if len(D) else D
                if len(D):
                    D = (D.assign(unit=D[f"admin{n}_pcode"] if n else D["iso3"], date=D["valid_from"].dt.strftime("%Y-%m-%d"), variable=col)
                         .groupby(["unit", "date", "variable"], as_index=False)["value"].sum(min_count=1))
                compare(S, f"admin{n}", source, D, exp, keys=["unit", "date"], label=f"{source} admin{n} {col} (sum by unit and period)")
                RAW_CHECKED.update((f"admin{n}", source, v) for v in variables)

In [ ]:
def idu_used(fig):
    """IDMC rule: the 'Recommended figure' rows if any, else the latest 'Triangulation' figure per place."""
    role = fig["role"].fillna("").astype(str).str.lower()
    rec = fig[role.str.startswith("recommend")]
    if len(rec):
        return rec
    tri = fig[role.str.startswith("triang")]
    base = tri if len(tri) else fig
    place = base["locations_name"].fillna("").astype(str) + "|" + base["locations_coordinates"].fillna("").astype(str)
    created = pd.to_datetime(base["created_at"], errors="coerce", utc=True)
    return base.assign(_place=place, _created=created).sort_values("_created", na_position="first").groupby("_place").tail(1)


def idu_saved_and_cached(iso3, event_ids):
    """IDU figures: the ones saved for the country, plus those of `event_ids` in the cached dump, without repeats."""
    parts = [pd.DataFrame(load_json(f)) for f in sorted(COUNTRY_DIR.glob("idmc/idu_figures_*.json")) if not f.name.endswith(".meta.json")]
    if len(IDU) and event_ids:
        parts.append(IDU[pd.to_numeric(IDU["event_id"], errors="coerce").isin([int(i) for i in event_ids])])
    parts = [x for x in parts if len(x)]
    out = pd.concat(parts, ignore_index=True).drop_duplicates("id") if parts else pd.DataFrame()
    return out[out["iso3"] == iso3] if len(out) else out


def gidd_saved(kind):
    """GIDD figures (from the GeoJSON) or disaster events saved for the country."""
    if kind == "figures":
        parts = []
        for f in sorted(COUNTRY_DIR.glob("idmc/gidd_figures_*.json")):
            if not f.name.endswith(".meta.json"):
                parts.append(pd.DataFrame([{**(x.get("properties") or {}), "coordinates": (x.get("geometry") or {}).get("coordinates")}
                                           for x in load_json(f).get("features", [])]))
        parts = [x for x in parts if len(x)]
        return pd.concat(parts, ignore_index=True).drop_duplicates("ID") if parts else pd.DataFrame()
    parts = [pd.read_parquet(f) for f in sorted(COUNTRY_DIR.glob("idmc/gidd_events_*.parquet"))]
    parts = [x for x in parts if len(x)]
    return pd.concat(parts, ignore_index=True).drop_duplicates(["event_name", "year"]) if parts else pd.DataFrame()


def first_filled(*series):
    for s in series:
        for v in (s if s is not None else []):
            if not missing(v) and str(v).strip() != "":
                return v
    return None


def split(text):
    return [x.strip() for x in str(text or "").split(";")]


with block(S, "IDMC event"):
    links_file = Path("dataset") / "_long" / "event" / "event_sources.parquet"
    links = access.links(EVENT_KEY) if links_file.is_file() else pd.DataFrame()
    idmc_links = links[links["source"] == "idmc"] if len(links) else links
    ids = sorted(idmc_links.loc[idmc_links["primary"], "source_event_id"].astype(str)) if len(idmc_links) else []
    explain("IDMC events of this folder", primary=ids or "none",
            others=sorted(idmc_links.loc[~idmc_links["primary"], "source_event_id"].astype(str)) if len(idmc_links) else [])
    if not ids and rows("event", "idmc_idu").empty and rows("event", "idmc_gidd").empty:
        raise Skip("no IDMC event linked to this folder: its displacement stays missing")
    figs = idu_saved_and_cached(ISO3, ids)
    gfig = gidd_saved("figures")
    gev = gidd_saved("events")
    exp_idu, exp_gidd, exp_loc_idu, exp_loc_gidd = [], [], [], []
    for i in ids:
        f = figs[pd.to_numeric(figs["event_id"], errors="coerce") == int(i)] if len(figs) else pd.DataFrame()
        g = gfig[pd.to_numeric(gfig["Event ID"], errors="coerce") == int(i)] if len(gfig) else pd.DataFrame()
        name = first_filled(g.get("Event name"), f.get("event_name"))
        kind = first_filled(g.get("Hazard Type"), g.get("Hazard type"), f.get("type"))
        codes = first_filled(g.get("Event codes (Code:Type)"), g.get("Event codes"), f.get("event_codes"))
        if len(f):
            u = idu_used(f)
            explain(f"IDU event {i}", figures=len(f), used=len(u), rule="recommended" if u["role"].astype(str).str.lower().str.startswith("recommend").any() else "latest triangulation per place",
                    total=pd.to_numeric(u["figure"], errors="coerce").sum())
            exp_idu += [dict(variable="disp_flow", value=pd.to_numeric(u["figure"], errors="coerce").sum(min_count=1)),
                        dict(variable="disp_n_figures", value=len(f)), dict(variable="disp_n_figures_used", value=len(u)),
                        dict(variable="event_idmc_id", text=str(i)), dict(variable="event_name", text=name),
                        dict(variable="event_codes", text=codes), dict(variable="hazard_type_name", text=kind)]
            for r in u.to_dict("records"):
                names, coords = split(r.get("locations_name")), split(r.get("locations_coordinates"))
                acc, types = split(r.get("locations_accuracy")), split(r.get("locations_type"))
                term = re.search(r"displacements?\s*\(([^)]*)\)", str(r.get("standard_info_text")))
                for k, place in enumerate(names):
                    lat = lon = None
                    if k < len(coords) and "," in coords[k]:
                        lat, lon = (float(x) for x in coords[k].split(",")[:2])
                    base = dict(location_id=f"idmc_{r['id']}_{k + 1}")
                    exp_loc_idu += [dict(base, variable="disp_figure", value=r.get("figure")), dict(base, variable="location_name", text=place),
                                    dict(base, variable="location_lat", value=lat), dict(base, variable="location_lon", value=lon),
                                    dict(base, variable="location_accuracy", text=acc[k] if k < len(acc) else None),
                                    dict(base, variable="location_type", text=types[k] if k < len(types) else None),
                                    dict(base, variable="figure_role", text=r.get("role")), dict(base, variable="figure_qualifier", text=r.get("qualifier")),
                                    dict(base, variable="figure_term", text=term.group(1) if term else None),
                                    dict(base, variable="figure_sources", text=r.get("sources")),
                                    dict(base, variable="figure_displacement_occurred", text=r.get("displacement_occurred"))]
        if len(g):
            cat = g["Figure category"].astype(str).str.strip().str.lower()
            flows = g[cat.str.startswith("internal displacement")]
            stock = g[cat.isin(["total number of idps", "idps"]) | cat.str.startswith("total number of idp")]
            exp_gidd += [dict(variable="event_idmc_id", text=str(i)), dict(variable="event_name", text=name),
                         dict(variable="hazard_type_name", text=kind)]
            if len(flows):
                exp_gidd.append(dict(variable="disp_flow", value=pd.to_numeric(flows["Total figures"], errors="coerce").sum(min_count=1)))
            if len(stock):
                y = pd.to_numeric(stock["Year"], errors="coerce").max()
                exp_gidd.append(dict(variable="disp_stock", value=pd.to_numeric(stock.loc[pd.to_numeric(stock["Year"], errors="coerce") == y, "Total figures"], errors="coerce").sum(min_count=1)))
            names_g = set(g["Event name"].dropna())
            if len(gev) and names_g:
                d = gev[gev["event_name"].isin(names_g)]
                if len(d):
                    exp_gidd.append(dict(variable="disp_flow_disasters_table", value=pd.to_numeric(d["new_displacement"], errors="coerce").sum(min_count=1)))
            for r in g.to_dict("records"):
                places = r.get("Locations name") or []
                coords = r.get("coordinates") or []

                def item(field, k):
                    v = r.get(field)
                    return v[k] if isinstance(v, list) and k < len(v) else None

                for k, place in enumerate(places or [None]):
                    lon, lat = coords[k][:2] if k < len(coords) and len(coords[k]) >= 2 else (None, None)
                    base = dict(location_id=f"gidd_{r.get('ID')}_{k + 1}")
                    exp_loc_gidd += [dict(base, variable="disp_figure", value=r.get("Total figures")),
                                     dict(base, variable="figure_reported", value=r.get("Reported figures")),
                                     dict(base, variable="figure_household_size", value=r.get("Household size")),
                                     dict(base, variable="location_name", text=place), dict(base, variable="location_lat", value=lat),
                                     dict(base, variable="location_lon", value=lon),
                                     dict(base, variable="location_accuracy", text=item("Locations accuracy", k)),
                                     dict(base, variable="location_type", text=item("Locations type", k)),
                                     dict(base, variable="location_pcode", text=item("Pcode", k)),
                                     dict(base, variable="figure_category", text=r.get("Figure category")),
                                     dict(base, variable="figure_housing_destruction", text=r.get("Is housing destruction"))]
    for level, source, D, exp, keys in (("event", "idmc_idu", rows("event", "idmc_idu"), exp_idu, []),
                                        ("event", "idmc_gidd", rows("event", "idmc_gidd"), exp_gidd, []),
                                        ("location", "idmc_idu", rows("location", "idmc_idu"), exp_loc_idu, ["location_id"]),
                                        ("location", "idmc_gidd", rows("location", "idmc_gidd"), exp_loc_gidd, ["location_id"])):
        with block(S, f"{source} {level}"):
            compare(S, level, source, D, exp, keys=keys, label=f"{source} {level}")

with block(S, "IDMC country"):
    exp = []
    for f in sorted(COUNTRY_DIR.glob("idmc/gidd_country_*")):
        if f.name.endswith(".meta.json") or f.suffix not in (".parquet", ".json"):
            continue
        if f.suffix == ".json" and f.with_suffix(".parquet").is_file():
            continue
        t = pd.read_parquet(f) if f.suffix == ".parquet" else pd.DataFrame(load_json(f))
        for r in t.to_dict("records"):
            for cause in ("conflict", "disaster"):
                exp += [dict(reason=cause, year=int(r["year"]), variable="disp_flow", value=r.get(f"{cause}_new_displacement")),
                        dict(reason=cause, year=int(r["year"]), variable="disp_stock", value=r.get(f"{cause}_total_displacement"))]
    D = rows("admin0", "idmc_gidd")
    D = D.assign(year=D["valid_to"].dt.year) if len(D) else D
    with block(S, "GIDD country"):
        compare(S, "admin0", "idmc_gidd", D, exp, keys=["reason", "year"], label="GIDD country")
    with block(S, "IDU country history"):
        if IDU.empty:
            raise Skip("no IDU dump")
        h = IDU[IDU["iso3"] == ISO3]
        used = pd.concat([idu_used(g) for _, g in h.groupby("event_id", dropna=False)], ignore_index=True) if len(h) else h
        exp = []
        if len(used):
            day = pd.to_datetime(used["displacement_date"], errors="coerce").fillna(pd.to_datetime(used["displacement_start_date"], errors="coerce"))
            year = day.dt.year.fillna(pd.to_numeric(used.get("year"), errors="coerce"))
            t_ = used["displacement_type"].astype(str).str.lower()
            cause = np.where(t_.str.contains("conflict"), "conflict", np.where(t_.str.contains("disaster"), "disaster", None))
            u = used.assign(_year=year, _cause=cause, _value=pd.to_numeric(used["figure"], errors="coerce")).dropna(subset=["_year", "_cause"])
            for (y, c), g in u.groupby(["_year", "_cause"]):
                exp += [dict(reason=c, year=int(y), variable="disp_flow", value=g["_value"].sum(min_count=1)),
                        dict(reason=c, year=int(y), variable="disp_events", value=g["event_id"].nunique())]
        D = rows("admin0", "idmc_idu")
        D = D.assign(year=D["valid_from"].dt.year) if len(D) else D
        compare(S, "admin0", "idmc_idu", D, exp, keys=["reason", "year"], label="IDU country history")

In [ ]:
with block(S, "Montandon"):
    folder = EVENT_DIR / "montandon"
    if not folder.is_dir():
        raise Skip("nothing saved from Montandon")
    exp = []
    if (folder / "impacts.parquet").is_file():
        t = pd.read_parquet(folder / "impacts.parquet")
        t = t[~t["duplicate"].astype(bool)] if "duplicate" in t else t
        exp += [dict(source_version=str(r["impact_id"]), variable="montandon_impact", value=r["value"], valid_from=r["date"]) for r in t.to_dict("records")]
    if (folder / "hazards.parquet").is_file():
        t = pd.read_parquet(folder / "hazards.parquet")
        exp += [dict(source_version=str(r["hazard_id"]), variable="montandon_hazard_severity", value=r["severity_value"], valid_from=r["date"])
                for r in t.to_dict("records")]
    compare(S, "event", "montandon", rows("event", "montandon"), exp, keys=["source_version"], label="Montandon")

with block(S, "ReliefWeb"):
    f = EVENT_DIR / "reliefweb" / "reliefweb.json"
    if not f.is_file():
        raise Skip("no ReliefWeb file")
    data = load_json(f)
    exp = []
    for r in data.get("reports", []):
        days = r.get("days_after_event")
        exp.append(dict(source_version=str(r.get("reliefweb_id")), variable="report_published", value=days,
                        text=None if is_number(days) else r.get("title"), valid_from=(r.get("date") or {}).get("original"),
                        known_at=(r.get("date") or {}).get("created")))
    D = rows("event", "reliefweb")
    compare(S, "event", "reliefweb", D[D["variable"] == "report_published"] if len(D) else D, exp, keys=["source_version"], label="ReliefWeb reports")
    M = D[D["variable"] == "disp_mention"] if len(D) else D
    if len(M):
        ok = [str(int(v)) in re.sub(r"[,.\s]", "", str(t)) for v, t in zip(M["value"], M["value_text"])]
        check(S, "ReliefWeb.disp_mention: every number is in its passage", all(ok), f"{sum(ok)} of {len(ok)}")
        RAW_CHECKED.add(("event", "reliefweb", "disp_mention"))

### 14b - Every number is in the file it comes from

A check that knows nothing about the sources: every number of the long tables whose `source_file`
is a JSON file must be written in that file (numbers inside text count too). Values computed from
several numbers (counts, sums, the rounded card exposure) cannot be found as such: they are listed
as computed and were rebuilt in 14a. Rows with `nature = derived` are left out.

In [ ]:
S = "14b provenance"
COMPUTED = {"event_n_episodes", "location_n_polygons", "exposure_pop_mmi_ge_7", "exposure_pop_mmi_ge_9", "media_news_count",
            "media_source_count", "disp_flow", "disp_stock", "disp_events", "disp_n_figures", "disp_n_figures_used",
            "disp_flow_disasters_table", "location_area_km2"}
with block(S, "numbers found in their files"):
    frames = [rows(lv) for lv in (*EVENT_LEVELS, *ADMIN_LEVELS)]
    allr = pd.concat([f for f in frames if len(f)], ignore_index=True)
    num = allr[allr["value"].notna() & allr["source_file"].fillna("").str.endswith(".json") & (allr["nature"] != "derived")]
    summary = []
    for (level, source, variable), g in num.groupby(["level", "source", "variable"]):
        big = {f for f in g["source_file"].unique() if not Path(f).is_file() or size_of(f) > MAX_PROVENANCE_MB * 1e6}
        g2 = g[~g["source_file"].isin(big)]
        found = [in_file(v, f) for v, f in zip(g2["value"], g2["source_file"])]
        name = f"{level}/{source}.{variable}"
        summary.append({"level": level, "source": source, "variable": variable, "values": len(g), "searched": len(g2), "found": sum(found)})
        if not len(g2):
            record(S, name, "SKIP", f"file missing or larger than {MAX_PROVENANCE_MB} MB: {sorted(big)[:2]}")
        elif variable in COMPUTED:
            record(S, name, "PASS", f"computed, rebuilt in 14a ({sum(found)} of {len(g2)} also written as such)")
        else:
            files = ", ".join(sorted({Path(f).name for f in g2["source_file"]})[:3])
            check(S, name, all(found), f"{sum(found)} of {len(g2)} found in {files}")
    display(pd.DataFrame(summary))

### 14c - Long tables against the tables of the country

The rules of the tables in `dataset/<ISO3>/<YEAR>/`, written again here:

- **event, episode, location**: one column per variable; when two sources give the same variable each
  gets its own column with the source name; the value known last wins. Extra columns: the last ShakeMap
  version (`_last`), the number of versions, the last model run of a ShakeMap, ReliefWeb counts,
  Montandon figures (not those found only by country and dates).
- **admin0, admin1, admin2**: the last value valid by 31 December of the year (sum or mean where the rule says so).

In [ ]:
S = "14c long -> tables"
SHORT = {"idmc_idu": "idu", "idmc_gidd": "gidd", "iom_dtm": "dtm", "hdx_idps": "hapi"}
EVENT_SOURCES = ("gdacs", "emdat", "emm", "idmc_idu", "idmc_gidd")
LOCATION_SOURCES = ("gdacs", "ghsl_pop", "ghsl_smod", "ghsl_pop_smod", "idmc_idu", "idmc_gidd")
LAST_RUN = ("impact_model_population_over_vii", "impact_model_max_mmi", "impact_model_alert_score")
MONTANDON_TYPES = ("displaced_internal", "displaced_total", "death", "injured", "missing", "affected_total", "homeless", "evacuated")
ID_COLUMNS = {"iso3", "year", "hazard_type", "event_id", "episode_id", "location_id", "admin1_pcode", "admin1_name",
              "admin2_pcode", "admin2_name"}


def cell_value(df):
    return df["value"].astype(object).where(df["value"].notna(), df["value_text"])


def suffixed(df):
    """A variable given by two sources: one column each, with the short source name."""
    df = df.copy()
    many = df.groupby("column")["source"].transform("nunique") > 1
    df.loc[many, "column"] = df.loc[many, "column"] + "_" + df.loc[many, "source"].map(lambda s: SHORT.get(s, s))
    return df


def latest(df, keys):
    """Value known last for each unit and column; no known_at = oldest; same known_at = the later row."""
    df = df.assign(_cell=cell_value(df), _order=np.arange(len(df)), _known=df["known_at"].fillna(pd.Timestamp("1900-01-01")))
    df = df.sort_values(["_known", "_order"])
    for k in keys:
        df[k] = df[k].map(_key)
    return df.groupby([*keys, "column"])["_cell"].last()


def compare_table(level, table, expected, keys):
    """Every cell of a table against the value rebuilt from the long tables."""
    t = table.copy()
    for k in keys:
        t[k] = t[k].map(_key)
    t = t.set_index(keys)
    exp_cols = set(expected.index.get_level_values("column"))
    cols = [c for c in t.columns if c not in ID_COLUMNS]
    extra, lacking = sorted(set(cols) - exp_cols), sorted(exp_cols - set(cols))
    check(S, f"{level}: every column comes from the long tables", not extra, f"unexpected: {extra[:8]}" if extra else f"{len(cols)} columns")
    check(S, f"{level}: every value of the long tables has its column", not lacking, f"missing: {lacking[:8]}" if lacking else "")
    bad_cols = {}
    for c in sorted(set(cols) & exp_cols):
        e = expected.xs(c, level="column")
        got = t[c]
        bad = []
        for k in e.index.union(got.dropna().index):
            want, have = e.get(k), got.get(k)
            if missing(want) and missing(have):
                continue
            if missing(want) or missing(have) or not equal(have, want):
                bad.append((k, have, want))
        if bad:
            bad_cols[c] = bad
        else:
            VERIFIED[(level, c)] = "long -> table"
    for c, bad in bad_cols.items():
        check(S, f"{level}.{c}", False, f"{len(bad)} cells differ (unit, table, expected): {bad[:3]}")
    check(S, f"{level}: cells equal to the long tables", not bad_cols,
          f"{len(set(cols) & exp_cols) - len(bad_cols)} of {len(set(cols) & exp_cols)} columns equal, {len(t)} rows")


with block(S, "event table"):
    if "event" not in T:
        raise Skip("no event table")
    ids = set(T["event"]["event_id"])
    ev = LONG["event"][LONG["event"]["event_id"].isin(ids)]
    parts = [suffixed(ev[ev["source"].isin(EVENT_SOURCES)].assign(column=lambda d: d["variable"]))]
    extra = {}
    shake = ev[ev["source"] == "gdacs_shakemap"]
    if len(shake):
        last = shake[shake["known_at"] == shake.groupby("event_id")["known_at"].transform("max")]
        parts.append(last.assign(column=last["variable"] + "_last"))
        extra["shakemap_n_versions"] = shake.groupby("event_id")["source_version"].nunique()
    imp = ev[(ev["source"] == "gdacs_shakemap_impact") & ev["variable"].isin(LAST_RUN)]
    if len(imp):
        last = imp[imp["known_at"] == imp.groupby("event_id")["known_at"].transform("max")]
        parts.append(last.assign(column=last["variable"] + "_last_shakemap"))
    reports = ev[ev["variable"] == "report_published"]
    if len(reports):
        extra["reliefweb_n_reports"] = reports.groupby("event_id").size()
        sit = reports[reports["reason"].fillna("").str.contains("Situation Report")]
        if len(sit):
            extra["reliefweb_n_situation_reports"] = sit.groupby("event_id").size()
    monty = ev[(ev["variable"] == "montandon_impact") & (ev["sector"] == "people") & ev["reason"].isin(MONTANDON_TYPES)
               & ~ev["quality_flag"].fillna("").str.contains("window_only")]
    if len(monty):
        monty = monty.sort_values("valid_from", kind="stable")
        parts.append(monty.assign(column="montandon_" + monty["reason"] + "_" + monty["provider"].str.replace("-impacts", "", regex=False).map(slug)))
    for variable, column in (("kg_node_count", "kg_n_nodes"), ("kg_edge_count", "kg_n_edges")):
        kg = ev[ev["variable"] == variable]
        if len(kg):
            extra[column] = kg.groupby("event_id")["value"].sum()
    expected = latest(pd.concat(parts, ignore_index=True), ["event_id"])
    more = [pd.Series(s.values, index=pd.MultiIndex.from_arrays([[_key(i) for i in s.index], [c] * len(s)], names=["event_id", "column"]))
            for c, s in extra.items()]
    expected = pd.concat([expected, *more]) if more else expected
    compare_table("event", T["event"], expected, ["event_id"])

with block(S, "episode table"):
    if "episode" not in T:
        raise Skip("no episode table")
    ids = set(T["event"]["event_id"]) if "event" in T else {EVENT_KEY}
    ep = LONG["episode"][LONG["episode"]["event_id"].isin(ids)].copy()
    ep["column"] = ep["variable"]
    imp = ep["source"] == "gdacs_impact"
    ep.loc[imp, "column"] = [f"{slug(g)}_{r}" for g, r in zip(ep.loc[imp, "population_group"], ep.loc[imp, "reason"])]
    compare_table("episode", T["episode"], latest(suffixed(ep), ["event_id", "episode_id"]), ["event_id", "episode_id"])

with block(S, "location table"):
    if "location" not in T:
        raise Skip("no location table")
    ids = set(T["event"]["event_id"]) if "event" in T else {EVENT_KEY}
    loc = LONG["location"][LONG["location"]["event_id"].isin(ids) & LONG["location"]["source"].isin(LOCATION_SOURCES)].copy()
    loc["column"] = [v.replace("_class", f"_{g}") if v.endswith("_class") and not missing(g) else v
                     for v, g in zip(loc["variable"], loc["population_group"])]
    compare_table("location", T["location"], latest(suffixed(loc), ["event_id", "episode_id", "location_id"]),
                  ["event_id", "episode_id", "location_id"])

In [ ]:
ALL_ADMIN = ADMIN_LEVELS
UNIT_KEY = {"admin0": "iso3", "admin1": "admin1_pcode", "admin2": "admin2_pcode"}


def where(**wanted):
    def f(d):
        m = pd.Series(True, index=d.index)
        for k, v in wanted.items():
            m &= d[k] == v
        return m
    return f


def needs(status):
    return where(sector="Intersectoral", population_group="total", gender="all", age_range="all", population_status=status)


ADMIN_RULES = [  # column, source, variable, which rows, how, levels
    *[(f"{v}_{cause}_{short}", src, v, where(reason=cause), "last", ("admin0",))
      for src, short, variables in (("idmc_gidd", "gidd", ("disp_flow", "disp_stock")), ("idmc_idu", "idu", ("disp_flow", "disp_events")))
      for v in variables for cause in ("disaster", "conflict")],
    ("idp_stock_dtm", "iom_dtm", "idp_stock", where(gender="all"), "sum", ALL_ADMIN),
    ("idp_stock_natural_disaster_dtm", "iom_dtm", "idp_stock",
     lambda d: (d["gender"] == "all") & d["reason"].fillna("").str.contains("Natural disaster"), "sum", ALL_ADMIN),
    ("needs_pin", "hdx_needs", "needs_people", needs("INN"), "last", ALL_ADMIN),
    ("needs_targeted", "hdx_needs", "needs_people", needs("TGT"), "last", ALL_ADMIN),
    ("pop_hno", "hdx_needs", "pop_hno", needs("all"), "last", ALL_ADMIN),
    ("refugees_hosted", "hdx_refugees", "refugee_population",
     lambda d: (d["gender"] == "all") & (d["age_range"] == "all") & d["population_group"].isin(["REF", "ASY", "OIP"])
     & (d["destination_code"] == ISO3) & (d["origin_code"] != ISO3), "sum", ("admin0",)),
    ("refugees_abroad", "hdx_refugees", "refugee_population",
     lambda d: (d["gender"] == "all") & (d["age_range"] == "all") & d["population_group"].isin(["REF", "ASY", "OIP"])
     & (d["origin_code"] == ISO3) & (d["destination_code"] != ISO3), "sum", ("admin0",)),
    ("returnees_refugees", "hdx_returnees", "returnee_population", where(gender="all", age_range="all", population_group="RET"), "sum", ("admin0",)),
    ("returnees_idps", "hdx_returnees", "returnee_population", where(gender="all", age_range="all", population_group="RDP"), "sum", ("admin0",)),
    *[(v, "hdx_poverty", v, None, "last", ("admin0", "admin1"))
      for v in ("poverty_mpi", "poverty_headcount_ratio", "poverty_intensity", "poverty_vulnerable", "poverty_severe")],
]


def admin_expected(level):
    key = UNIT_KEY[level]
    df = LONG[level]
    if df.empty:
        return {}, {}
    df = df[(df["iso3"] == ISO3) & (df["valid_from"] <= pd.Timestamp(f"{YEAR}-12-31"))]
    rules = [r for r in ADMIN_RULES if level in r[5]]
    if level == "admin0":
        rules += [(v, "inform", v, None, "last", (level,)) for v in sorted(df.loc[df["source"] == "inform", "variable"].unique())]
    gh = df[df["source"].isin(("ghsl_pop", "ghsl_smod", "ghsl_pop_smod"))][["source", "variable", "population_group"]].drop_duplicates()
    for r in gh.itertuples():
        if missing(r.population_group):
            rules.append((r.variable, r.source, r.variable, lambda d: d["population_group"].isna(), "last", (level,)))
        else:
            rules.append((r.variable.replace("_class", f"_{r.population_group}") if r.variable.endswith("_class") else r.variable,
                          r.source, r.variable, where(population_group=r.population_group), "last", (level,)))
    for good in sorted(df.loc[df["variable"] == "food_price", "commodity"].dropna().unique()):
        rules.append((f"food_price_{slug(good)}", "hdx_food_prices", "food_price", where(commodity=good), "mean", (level,)))
    out, ties = {}, {}
    for col, src, var, which, how, _ in rules:
        r = df[(df["source"] == src) & (df["variable"] == var)]
        if len(r) and which is not None:
            r = r[which(r)]
        r = r[r[key].notna()]
        if r.empty:
            continue
        if how == "mean":
            r = r[(r["valid_from"] <= pd.Timestamp(f"{YEAR}-12-31")) & (r["valid_to"] >= pd.Timestamp(f"{YEAR}-01-01"))]
            if r.empty:
                continue
            out[col] = r.groupby(key)["value"].mean()
            continue
        end = r["valid_to"].fillna(r["valid_from"])
        r = r[end == end.groupby(r[key]).transform("max")]
        if how == "sum":
            out[col] = r.groupby(key)["value"].sum(min_count=1)
        else:
            cells = r.assign(_cell=cell_value(r))
            out[col] = cells.groupby(key)["_cell"].first()
            ties[col] = cells.groupby(key)["_cell"].agg(lambda s: {str(x) for x in s})
    return out, ties


for level in ADMIN_LEVELS:
    with block(S, f"{level} table"):
        if level not in T:
            raise Skip(f"no {level} table")
        key = UNIT_KEY[level]
        t = T[level].set_index(key)
        out, ties = admin_expected(level)
        cols = [c for c in t.columns if c not in ID_COLUMNS]
        extra, lacking = sorted(set(cols) - set(out)), sorted(set(out) - set(cols))
        check(S, f"{level}: every column follows a rule", not extra, f"unexpected: {extra[:8]}" if extra else f"{len(cols)} columns")
        check(S, f"{level}: every rule with values has its column", not lacking, f"missing: {lacking[:8]}" if lacking else "")
        bad_cols, tied, lost = {}, [], set()
        for col, vals in out.items():
            if col not in t:
                continue
            got = t[col]
            bad = []
            for unit, want in vals.items():
                if unit not in got.index:
                    lost.add(unit)
                    continue
                have = got[unit]
                if missing(want) and missing(have):
                    continue
                if missing(want) or missing(have) or not equal(have, want):
                    if col in ties and str(have) in ties[col].get(unit, set()):
                        tied.append((col, unit))
                    else:
                        bad.append((unit, have, want))
            bad += [(u, got[u], None) for u in got.dropna().index if u not in vals.index]
            if bad:
                bad_cols[col] = bad
            else:
                VERIFIED[(level, col)] = "long -> table"
        for col, bad in bad_cols.items():
            check(S, f"{level}.{col}", False, f"{len(bad)} cells differ (unit, table, expected): {bad[:3]}")
        check(S, f"{level}: cells equal to the long tables", not bad_cols, f"{len(out) - len(bad_cols)} of {len(out)} columns equal, {len(t)} units")
        warn(S, f"{level}: one value when several rows end on the same day", not tied, f"{len(tied)} cells took one of several values: {tied[:4]}" if tied else "")
        warn(S, f"{level}: every unit of the long tables is in the boundaries", not lost, f"left out: {sorted(lost)[:6]}" if lost else "")

with block(S, "columns.csv"):
    if COLS.empty:
        raise Skip("no columns.csv")
    for level, t in T.items():
        meta = COLS[COLS["level"] == level]
        cols = [c for c in t.columns if c not in ID_COLUMNS]
        undescribed = sorted(set(cols) - set(meta["column"]))
        check(S, f"{level}: every column is in columns.csv", not undescribed, undescribed[:6] or f"{len(cols)} columns")
        wrong = [c for c, n in zip(meta["column"], meta["n_values"]) if c in t and int(t[c].notna().sum()) != int(n)]
        check(S, f"{level}: n_values of columns.csv", not wrong, wrong[:6] or "")
        back = meta[pd.to_datetime(meta["valid_to"]) < pd.to_datetime(meta["valid_from"])]
        check(S, f"{level}: validity of the columns runs forward", back.empty, list(back["column"])[:6] or "")

### 14d - Grid cells against the areas

The grid keeps one row per cell of every area. Summed by area, it must give the area values.
Cells without people are not saved in the 100 m grid: totals do not change.

In [ ]:
S = "14d grid"
with block(S, "100 m cells"):
    try:
        g = access.grid(event_id=EVENT_KEY, resolution_m=100)
    except FileNotFoundError as e:
        raise Skip(str(e))
    loc = rows("location", "ghsl_pop")
    check(S, "only cells with people", bool((g["pop_ghsl"] > 0).all()), f"{len(g):,} cells")
    check(S, "coverage between 0 and 1", bool(g["coverage"].between(0, 1 + 1e-9).all()))
    for (ep, lid), cells in g.groupby(["episode_id", "location_id"]):
        r = loc[(loc["episode_id"] == ep) & (loc["location_id"] == lid)]
        pop = r.loc[r["variable"] == "pop_ghsl", "value"]
        cov = r.loc[r["variable"] == "pop_ghsl_covered", "value"]
        same(S, f"{lid}: people of the cells = pop_ghsl", cells["pop_ghsl"].sum(), pop.iloc[0] if len(pop) else None)
        same(S, f"{lid}: weighted people of the cells = pop_ghsl_covered", cells["pop_ghsl_covered"].sum(), cov.iloc[0] if len(cov) else None, rel=1e-3)
    VERIFIED.update({("grid", "pop_ghsl"): "grid -> location", ("grid", "pop_ghsl_covered"): "grid -> location"})

with block(S, "1 km cells"):
    try:
        g = access.grid(event_id=EVENT_KEY, resolution_m=1000)
    except FileNotFoundError as e:
        raise Skip(str(e))
    loc = rows("location", "ghsl_pop_smod")
    for (ep, lid), cells in g.groupby(["episode_id", "location_id"]):
        r = loc[(loc["episode_id"] == ep) & (loc["location_id"] == lid) & (loc["variable"] == "smod_cells_class")]
        want = {int(k): int(v) for k, v in zip(r["population_group"], r["value"])}
        have = {int(k): int(v) for k, v in cells["smod_class"].round().astype(int).value_counts().items()}
        check(S, f"{lid}: 1 km cells by class = smod_cells_<class>", want == have, f"grid {have}, location {want}")
    VERIFIED[("grid", "smod_class")] = "grid -> location"

### 14e - Consistency

Totals, shares, dates and versions that must agree inside the dataset.

In [ ]:
S = "14e consistency"
with block(S, "dates and units"):
    for lv in (*EVENT_LEVELS, *ADMIN_LEVELS):
        df = rows(lv)
        if df.empty:
            continue
        back = df[df["valid_to"] < df["valid_from"]]
        check(S, f"{lv}: validity runs forward", back.empty, f"{len(back)} of {len(df)} rows")
        check(S, f"{lv}: every value has source, start and nature",
              bool(df["source"].notna().all() and df["valid_from"].notna().all() and df["nature"].notna().all()))
        people = df[df["unit"] == "people"]
        check(S, f"{lv}: no negative people", int((people["value"] < 0).sum()) == 0, f"{len(people)} rows")
        shares = df[df["unit"] == "share"]
        check(S, f"{lv}: shares between 0 and 1", bool(shares["value"].between(-1e-9, 1 + 1e-9).all()), f"{len(shares)} rows")
    gd = rows("event", "gdacs")
    if len(gd):
        check(S, "GDACS known_at (datemodified) after the event start", bool((gd["known_at"] >= gd["valid_from"]).all()))

with block(S, "ShakeMap versions"):
    versions = P.get("shakemap") or []
    if not versions:
        raise Skip("no ShakeMap")
    last = [v for v in versions if v.get("last")]
    newest = max(versions, key=lambda v: str(v.get("dateinserted")))
    check(S, "one version marked last", len(last) == 1, [v.get("shakeid") for v in last])
    check(S, "the version marked last is the newest", bool(last) and last[0].get("shakeid") == newest.get("shakeid"),
          f"last {[v.get('shakeid') for v in last]}, newest {newest.get('shakeid')} ({newest.get('dateinserted')})")

with block(S, "GHSL totals of each area"):
    loc = rows("location")
    gh = loc[loc["source"].astype(str).str.startswith("ghsl")] if len(loc) else loc
    if gh.empty:
        raise Skip("no GHSL area")
    for (ep, lid), r in gh.groupby(["episode_id", "location_id"]):
        def get(variable, group=None):
            x = r[(r["variable"] == variable) & ((r["population_group"] == group) if group else r["population_group"].isna())]["value"]
            return float(x.iloc[0]) if len(x) else 0.0
        pop = get("pop_ghsl")
        flags = r.loc[r["source"] == "ghsl_pop_smod", "quality_flag"].dropna().astype(str)
        share = float(re.search(r"no_smod_cell_share=([\d.]+)", flags.iloc[0]).group(1)) if len(flags) and "no_smod_cell_share" in flags.iloc[0] else 0.0
        if (r["source"] == "ghsl_pop_smod").any():
            parts = get("pop_ghsl_rural") + get("pop_ghsl_urban") + get("pop_ghsl_class", "10") + share * pop
            same(S, f"{lid}: rural + urban + water + no class = pop_ghsl", parts, pop, rel=1e-3)
        check(S, f"{lid}: weighted people <= people of the touched cells", get("pop_ghsl_covered") <= pop + 1e-6,
              f"{get('pop_ghsl_covered'):,.0f} <= {pop:,.0f}")
        fr = r[r["variable"] == "smod_area_share_class"]["value"]
        if len(fr):
            warn(S, f"{lid}: area shares by class add up to 1", abs(fr.sum() - 1) < 1e-3, f"{fr.sum():.6f}")

with block(S, "impact runs: population over VII"):
    imp = rows("episode", "gdacs_impact")
    if imp.empty:
        raise Skip("no impact run")
    for f, r in imp.groupby("source_file"):
        over = r.loc[r["population_group"] == "PopulationOverVII", "value"]
        eps = r[r["population_group"].astype(str).str.fullmatch(r"ep\d+")]
        high = eps[eps["population_group"].str[2:].astype(int) >= 7]["value"].sum()
        if len(over):
            warn(S, f"{Path(f).name}: PopulationOverVII = ep7 + ep8 + ...", equal(over.iloc[0], high),
                 f"{over.iloc[0]:,.0f} against {high:,.0f} ({', '.join(eps['population_group'])})")

with block(S, "intensity bands do not overlap"):
    import geopandas as gpd
    files = sorted((EVENT_DIR / "episodes" / str(MAIN_EPISODE) / "areas").glob("Intensity_*/area.geojson"))
    if not files:
        raise Skip("no intensity band")
    bands = gpd.GeoDataFrame(pd.concat([gpd.read_file(f) for f in files], ignore_index=True)).to_crs("ESRI:54009")
    bands["geometry"] = bands.geometry.buffer(0)
    total, union = bands.area.sum(), bands.unary_union.area
    warn(S, "sum of the band areas = area of their union", abs(total - union) <= 0.01 * union,
         f"sum {total / 1e6:,.1f} km2, union {union / 1e6:,.1f} km2: the bands are rings, not nested areas")

with block(S, "areas: project formula against WGS84"):
    loc = rows("location", "gdacs")
    a = loc[loc["variable"] == "location_area_km2"] if len(loc) else loc
    if a.empty:
        raise Skip("no area")
    for r in a.to_dict("records"):
        f = EVENT_DIR / "episodes" / str(r["episode_id"]) / "areas" / r["location_id"] / "area.geojson"
        if not f.is_file():
            continue
        ell = area_km2(load_json(f)["features"][0]["geometry"], WGS84)
        warn(S, f"{r['location_id']}: area against the WGS84 ellipsoid", ell and abs(r["value"] - ell) <= 0.01 * ell,
             f"{r['value']:,.1f} km2 (sphere), {ell:,.1f} km2 (WGS84)")

### 14f - Sources against each other

These are not errors: the sources measure different things, at different times, with different
models. A WARN marks a difference larger than expected, worth reading.

In [ ]:
S = "14f between sources"


def long_value(level, source, variable, **where_):
    r = rows(level, source)
    if r.empty:
        return None
    r = r[r["variable"] == variable]
    for k, v in where_.items():
        r = r[r[k].astype(str) == str(v)]
    if r.empty:
        return None
    r = r.sort_values("known_at", na_position="first")
    return r["value"].iloc[-1] if r["value"].notna().iloc[-1] else r["value_text"].iloc[-1]


with block(S, "magnitude"):
    mags = {"GDACS card": long_value("event", "gdacs", "hazard_magnitude"), "GDACS severity": long_value("event", "gdacs", "hazard_severity"),
            "GDACS episode": long_value("episode", "gdacs", "hazard_magnitude", episode_id=MAIN_EPISODE),
            "last ShakeMap": long_value("event", "gdacs_shakemap", "shakemap_magnitude"), "EM-DAT": long_value("event", "emdat", "hazard_magnitude")}
    mags = {k: float(v) for k, v in mags.items() if is_number(v)}
    display(pd.Series(mags, name="magnitude"))
    warn(S, "magnitudes within 0.3", max(mags.values()) - min(mags.values()) <= 0.3, mags)

with block(S, "epicentre"):
    lon0, lat0 = EV_JSON["geometry"]["coordinates"][:2]
    card = card_of(EVENT_DIR)
    cl = [first_number(x) for x in str(card.get("Lat/Lon") or "").split(",")]
    points = {"GDACS card": tuple(cl) if len(cl) == 2 and None not in cl else None,
              "last ShakeMap": (long_value("event", "gdacs_shakemap", "shakemap_lat"), long_value("event", "gdacs_shakemap", "shakemap_lon")),
              "EM-DAT": (long_value("event", "emdat", "hazard_lat"), long_value("event", "emdat", "hazard_lon"))}
    for name, pt in points.items():
        if pt and all(is_number(x) for x in pt):
            km = haversine_km(lat0, lon0, float(pt[0]), float(pt[1]))
            warn(S, f"{name} epicentre within 30 km of GDACS", km <= 30, f"{km:.1f} km")

with block(S, "GHSL against the GDACS impact model"):
    imp = rows("episode", "gdacs_impact")
    imp = imp[imp["episode_id"] == str(MAIN_EPISODE)] if len(imp) else imp
    if imp.empty:
        raise Skip("no impact run")
    run = imp[imp["known_at"] == imp["known_at"].max()]
    run = run[run["source_file"] == run["source_file"].iloc[-1]]
    gh = rows("location", "ghsl_pop")
    gh = gh[(gh["episode_id"] == str(MAIN_EPISODE)) & (gh["variable"] == "pop_ghsl_covered")] if len(gh) else gh
    band = {float(l.split("_")[1]): v for l, v in zip(gh["location_id"], gh["value"]) if re.fullmatch(r"Intensity_[\d.]+", l)} if len(gh) else {}
    areas = sorted(float(f.parent.name.split("_")[1]) for f in (EVENT_DIR / "episodes" / str(MAIN_EPISODE) / "areas").glob("Intensity_*/area.geojson"))
    table = []
    for k in sorted(int(g[2:]) for g in run["population_group"] if re.fullmatch(r"ep\d+", str(g))):
        model = run.loc[run["population_group"] == f"ep{k}", "value"].iloc[0]
        ours = [band.get(m) for m in (k - 0.5, float(k)) if m in areas]
        table.append({"MMI": k, "GDACS model ep": model, "GHSL bands": f"{k - 0.5:g} and {k:g}",
                      "GHSL people": sum(x for x in ours if x is not None) if ours and None not in ours else None})
    print(f"People by intensity, model run {Path(run['source_file'].iloc[0]).name} against GHSL {YEAR} in the GDACS bands:")
    display(pd.DataFrame(table))
    over = run.loc[run["population_group"] == "PopulationOverVII", "value"]
    wanted = [m for m in areas if m >= 6.5]
    if not wanted or any(m not in band for m in wanted):
        raise Skip(f"GHSL is missing for some bands >= 6.5 ({wanted}): run section 5")
    ghsl = sum(band[m] for m in wanted)
    ratio = ghsl / over.iloc[0] if len(over) and over.iloc[0] else None
    warn(S, "GHSL people in bands >= 6.5 against PopulationOverVII", ratio is not None and abs(ratio - 1) <= REL_SOURCES,
         f"GHSL {ghsl:,.0f}, GDACS {over.iloc[0]:,.0f}, ratio {ratio:.2f}" if ratio else "no PopulationOverVII")

with block(S, "card against the model"):
    card7 = long_value("event", "gdacs", "exposure_pop_mmi_ge_7")
    model = long_value("event", "gdacs_shakemap_impact", "impact_model_population_over_vii")
    if not (is_number(card7) and is_number(model)):
        raise Skip("no card exposure or no ShakeMap model run")
    warn(S, "card exposure (rounded) against the last ShakeMap model run", abs(float(card7) - float(model)) <= 0.05 * float(model),
         f"card {float(card7):,.0f}, model {float(model):,.0f}")

with block(S, "impact figures"):
    figures = []
    for name, var in (("deaths", "impact_deaths"), ("injured", "impact_injured"), ("affected", "impact_total_affected"), ("homeless", "impact_homeless")):
        figures.append({"source": "EM-DAT", "figure": name, "value": long_value("event", "emdat", var)})
    for src in ("idmc_idu", "idmc_gidd"):
        figures.append({"source": src, "figure": "displacement", "value": long_value("event", src, "disp_flow")})
    mo = rows("event", "montandon")
    if len(mo):
        mo = mo[(mo["variable"] == "montandon_impact") & (mo["sector"] == "people")]
        for r in mo.to_dict("records"):
            figures.append({"source": f"Montandon {r['provider']}" + (" (window only)" if "window_only" in str(r["quality_flag"]) else ""),
                            "figure": r["reason"], "value": r["value"]})
    display(pd.DataFrame(figures))
    deaths = long_value("event", "emdat", "impact_deaths")
    mdeaths = [f["value"] for f in figures if f["figure"] == "death" and "window" not in f["source"]]
    if is_number(deaths) and mdeaths:
        warn(S, "EM-DAT deaths among the Montandon death figures", any(equal(deaths, m) for m in mdeaths), f"EM-DAT {deaths}, Montandon {mdeaths}")
    record(S, "impact figures", "PASS", f"{len(figures)} figures shown")

### 14g - Before and after

The tables of the country before this run (copied in section 2) against the new ones.

In [ ]:
S = "14g before/after"
with block(S, "before and after"):
    before = LAST_BACKUP / "dataset_before" if LAST_BACKUP else None
    if before is None or not before.is_dir():
        raise Skip("no copy of the dataset before this run")
    for level in ("event", "episode", "location", "admin0", "admin1", "admin2"):
        f = before / f"{level}.csv"
        old = pd.read_csv(f, encoding="utf-8-sig") if f.is_file() else pd.DataFrame()
        new = T.get(level, pd.DataFrame())
        added, removed = sorted(set(new.columns) - set(old.columns)), sorted(set(old.columns) - set(new.columns))
        print(f"{level}: {old.shape} -> {new.shape}; {len(added)} new columns {added[:10]}; {len(removed)} removed {removed[:10]}")
    oldf = before / "event.csv"
    if oldf.is_file() and "event" in T:
        old = pd.read_csv(oldf, encoding="utf-8-sig")
        o, n = old[old["event_id"] == EVENT_KEY], T["event"][T["event"]["event_id"] == EVENT_KEY]
        if len(o) and len(n):
            changed = [{"column": c, "before": o[c].iloc[0], "now": n[c].iloc[0]} for c in sorted(set(o.columns) & set(n.columns))
                       if not (missing(o[c].iloc[0]) and missing(n[c].iloc[0])) and not equal(o[c].iloc[0], n[c].iloc[0])]
            print(f"\nValues of {EVENT_KEY} that changed ({len(changed)}): GDACS rewrites its records, the rest should not move")
            display(pd.DataFrame(changed))
    record(S, "before and after", "PASS", before.as_posix())

### 14h - By hand

Open the pages and compare a few values with the dataset.

In [ ]:
S = "14h by hand"
with block(S, "pages to open"):
    e = T["event"][T["event"]["event_id"] == EVENT_KEY].iloc[0] if "event" in T else pd.Series(dtype=object)
    pages = {"GDACS report": (P.get("url") or {}).get("report"),
             "USGS event page": f"https://earthquake.usgs.gov/earthquakes/eventpage/{P.get('sourceid')}" if P.get("sourceid") else None,
             "EM-DAT": f"https://public.emdat.be (search {e.get('event_emdat_id')})" if e.get("event_emdat_id") else None,
             "ReliefWeb": f"https://reliefweb.int/disaster (GLIDE {e.get('event_external_ids')})" if e.get("event_external_ids") else None}
    for k, v in pages.items():
        if v:
            print(f"{k:<16} {v}")
    wanted = [c for c in ("alert_level", "alert_score", "hazard_magnitude_gdacs", "hazard_magnitude", "hazard_depth", "exposure_pop_mmi_ge_7",
                          "shakemap_max_mmi_last", "impact_deaths", "impact_injured", "impact_total_affected", "disp_flow_idu",
                          "disp_flow_gidd") if c in e.index and not missing(e[c])]
    display(e[wanted].rename("value in the dataset").to_frame())
    record(S, "pages to open", "PASS", f"{len(wanted)} values to compare")

### 14i - Which columns were checked

For every column of the tables of this event and of the country:

- **original + table**: the long values behind it were compared with the original files (14a) and the cells with the long values (14c);
- **table only**: the cells agree with the long tables, but the long values were not rebuilt from the original files;
- **not verified**: no check covered it, or a check failed on it.

In [ ]:
S = "14i coverage"
with block(S, "coverage"):
    out = []
    for level, t in T.items():
        if level in EVENT_LEVELS:
            t = t[t["event_id"] == EVENT_KEY]
        meta = COLS[COLS["level"] == level].set_index("column") if len(COLS) else pd.DataFrame()
        for c in t.columns:
            if c in ID_COLUMNS or t[c].isna().all():
                continue
            src = meta["source"].get(c) if len(meta) else None
            var = meta["variable"].get(c) if len(meta) else None
            raw = (level, src, var) in RAW_CHECKED
            wide = (level, c) in VERIFIED
            out.append({"level": level, "column": c, "source": src, "variable": var,
                        "status": "original + table" if raw and wide else "table only" if wide else "not verified"})
    cov = pd.DataFrame(out)
    display(cov.groupby(["level", "status"]).size().unstack(fill_value=0))
    weak = cov[cov["status"] != "original + table"]
    if len(weak):
        print("\nColumns not fully verified:")
        display(weak)
    warn(S, "every column verified against the original files", weak.empty,
         f"{len(cov) - len(weak)} of {len(cov)} columns; not fully verified: {list(weak['column'])[:10]}")

## 15 - Results

In [ ]:
results = pd.DataFrame(RESULTS)
results.to_csv("test_event_results.csv", index=False, encoding="utf-8-sig")
print(results["status"].value_counts().to_dict())
display(results.groupby(["section", "status"]).size().unstack(fill_value=0))
for status in ("FAIL", "WARN", "SKIP"):
    part = results[results["status"] == status]
    if len(part):
        print(f"\n{status} ({len(part)}):")
        display(part)

**Reading the results**

- **FAIL** in sections 3 to 13: a download or a step does not work. Read the message, it says which file or call.
- **FAIL** in 14a or 14b: a value of the dataset is not the one in the original file. The line gives the variable, the key and both values.
- **FAIL** in 14c or 14d: the tables of the country, or the grid, do not follow from the long tables.
- **WARN**: read it. A source changed since the download, two sources disagree, or a column is not covered by any check.
- **SKIP**: the source has no data for this event or country, or a key is missing. The reason is printed.